# **TRADING MULTI-AGENT**

# **1. Environment Preparation and Base Architecture**

Environment Preparation and Base Architecture

**1.1 Set up Google Drive as your Persistent Data Lake**

In [1]:
from google.colab import drive
import os

# Set up Google Drive
drive.mount('/content/drive')

# Define the project's baseline route
BASE_PATH = '/content/drive/MyDrive/Trading_Agent_POCv1'
print(f"Base path configured in: {BASE_PATH}")

Mounted at /content/drive
Base path configured in: /content/drive/MyDrive/Trading_Agent_POCv1


**1.2 Installation of Key Dependencies**

In [2]:
# Install main libraries
!pip install yfinance duckdb playwright langgraph crewai pandas nest_asyncio -q

# Install the system binaries and dependencies for Playwright (Chromium)
!playwright install

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 976.4/976.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.5/111.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Install system dependencies required by Playwright browsers
# This addresses errors like 'libatk-1.0.so.0: cannot open shared object file'
!playwright install-deps

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://cli.github.com/packages stable InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,006 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,301 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,183 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/restricted 

**1.3 Creating the Folder Structure**

In [4]:
# Definition of subfolders
folders = [
    f"{BASE_PATH}/data/raw/bvc",
    f"{BASE_PATH}/data/raw/yahoo",
    f"{BASE_PATH}/data/processed",
    f"{BASE_PATH}/data/metadata",
    f"{BASE_PATH}/logs"
]

# Create directories if they do not exist
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"✓ List structure: {folder}")

✓ List structure: /content/drive/MyDrive/Trading_Agent_POCv1/data/raw/bvc
✓ List structure: /content/drive/MyDrive/Trading_Agent_POCv1/data/raw/yahoo
✓ List structure: /content/drive/MyDrive/Trading_Agent_POCv1/data/processed
✓ List structure: /content/drive/MyDrive/Trading_Agent_POCv1/data/metadata
✓ List structure: /content/drive/MyDrive/Trading_Agent_POCv1/logs


**1.4 Initialization of the Logging System and Asynchronous Patching**

In [5]:
import logging
import nest_asyncio

# Apply the patch to enable asynchronous playwright in Colab
nest_asyncio.apply()

# Configure the logger
log_file = f"{BASE_PATH}/logs/pipeline_execution.log"
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler() # Muestra los logs en la consola de Colab
    ]
)

logging.info("Logs environment and infrastructure successfully initialized.")

**1.5 Smoke Test" (Base Connectivity Smoke Test)**

In [6]:
import yfinance as yf
import duckdb

logging.info("Starting testing of Sprint 1 components...")

try:
    # 1. Test connectivity with Yahoo Finance (Quick download of a test ticker)
    test_df = yf.download("AAPL", period="1d", interval="1m")
    if not test_df.empty:
        logging.info("✓ Yahoo Finance test: SUCCESSFUL (Data received)")
    else:
        logging.warning("! Yahoo Finance test: No data received")

    # 2. Test initialization of the DuckDB Relational Database
    db_path = f"{BASE_PATH}/data/processed/trading_platform.db"
    conn = duckdb.connect(db_path)

    # Create a quick test table
    conn.execute("CREATE TABLE IF NOT EXISTS test_status (id INTEGER, status VARCHAR);")
    conn.execute("INSERT INTO test_status VALUES (1, 'Sprint 1 Filled');")

    res = conn.execute("SELECT status FROM test_status WHERE id = 1;").fetchone()
    logging.info(f"✓ DuckDB test: SUCCESSFUL (Result in DB): '{res[0]}')")

    conn.close()
    logging.info("=== SPRINT 1 SUCCESSFULLY COMPLETED: 100% Operational Environment ===")

except Exception as e:
    logging.error(f"✕ Error during environment verification: {str(e)}")

/tmp/ipykernel_1187/2916477844.py:8: FutureWarning: YF.download() has changed argument auto_adjust default to True
  test_df = yf.download("AAPL", period="1d", interval="1m")
[*********************100%***********************]  1 of 1 completed


# **2. Extraction and Ingestion: Raw**

The main objective is not just to "download data", but to guarantee an enterprise-level design that demonstrates operational resilience, immutability, and complete traceability through metadata and digital signatures (SHA256 Hashes)

**2.1 Definition of Investment Assets**

In [7]:
# =========================================================================
# Explicit definition of the 30 tickers in Yahoo Finance
# =========================================================================

# 15 Highly Tradable International Assets
YAHOO_TICKERS_INT = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META", "BABA", "NFLX", "AMD", "V", "MA", "DIS", "JPM", "COIN"]

# 15 Local Assets of the BVC (the '.CO' suffix is ​​mandatory for Yahoo Finance)
BVC_TICKERS_YAHOO = [
    "ECOPETROL.CL", "CIBEST.CL", "GRUPOAVAL.CL", "ISA.CL",
    "GRUPOARGOS.CL", "PFCEMARGOS.CL", "NUTRESA.CL", "CEMARGOS.CL",
    "GRUPOSURA.CL:", "CORFICOLCF.CL", "BOGOTA.CL", "CELSIA.CL",
    "ETB.CL", "MINEROS.CL", "CNEC.CL"
]

print(f"✓ Configured {len(YAHOO_TICKERS_INT)} international assets.")
print(f"✓ Configured {len(BVC_TICKERS_YAHOO)} Local assets of the BVC mapped to Yahoo Finance.")
print(f" Total active ingredients ready for intake: {len(YAHOO_TICKERS_INT) + len(BVC_TICKERS_YAHOO)}")

✓ Configured 15 international assets.
✓ Configured 15 Local assets of the BVC mapped to Yahoo Finance.
 Total active ingredients ready for intake: 30


**2.2 Raw Data Lake Writer**

In [8]:
import json
import hashlib
import os
from datetime import datetime, timezone

class RawDataLakeWriter:
    def __init__(self, base_path):
        self.base_path = base_path

    def save_raw_data(self, data_str, source_name, filename, source_url):
        """
        Save the original raw files, attaching mandatory audit metadata.
        Ensure data immutability and provenance (Data Lineage).
        """
        # Immutable temporary structure: raw/source/yyyy/mm/dd
        today = datetime.now(timezone.utc).strftime("%Y/%m/%d")
        folder_path = os.path.join(self.base_path, f"data/raw/{source_name}", today)
        os.makedirs(folder_path, exist_ok=True)

        # Calculate SHA256 Hash for future integrity validation
        sha256_hash = hashlib.sha256(data_str.encode('utf-8')).hexdigest()

        # 1. Save original raw file (CSV or JSON)
        file_dest = os.path.join(folder_path, filename)
        with open(file_dest, 'w', encoding='utf-8') as f:
            f.write(data_str)

       # 2. Generate the Audit Metadata Schema (Traceability)
        metadata = {
            "source_url": source_url,
            "downloaded_at": datetime.now(timezone.utc).isoformat() + "Z",
            "filename": filename,
            "sha256": sha256_hash,
            "extractor_user": "Colab_Agent_Pipeline",
            "file_size_bytes": len(data_str)
        }

        # Save metadata in a parallel JSON file
        meta_filename = filename.split('.')[0] + "_metadata.json"
        meta_dest = os.path.join(folder_path, meta_filename)
        with open(meta_dest, 'w', encoding='utf-8') as f:
            json.dump(metadata, f, indent=4)

        return file_dest, meta_dest

**2.3 Extraction via API (Yahoo Finance Ingestion)**

In [9]:
import yfinance as yf
import pandas as pd
import logging

def extract_yahoo_finance(tickers, writer):
    logging.info(f"Initiating hybrid ingestion from Yahoo Finance API...")

    for ticker in tickers:
        try:
            # We downloaded the 1 month history (enough for POC validation)
            df = yf.download(ticker, period="1mo", interval="1d")
            if df.empty:
                logging.warning(f"⚠ Ticker {ticker} no data returned.")
                continue

           # Convert to raw CSV string to respect immutable storage
            raw_csv = df.to_csv()
            filename = f"{ticker}_raw.csv"
            source_url = f"https://finance.yahoo.com/quote/{ticker}"

            # Persistence with SHA256 signature
            f_path, _ = writer.save_raw_data(raw_csv, "yahoo", filename, source_url)
            logging.info(f"✓ Yahoo ingested: {ticker} -> saved in {os.path.basename(f_path)}")

        except Exception as e:
            logging.error(f"✕ Critical failure to extract {ticker} from Yahoo: {str(e)}")

**2.4 Extraction of Local Assets via Yahoo Finance**

In [10]:
import yfinance as yf
import pandas as pd
import logging
import os

def extract_colombian_assets_via_yahoo(tickers_co, writer):
    """
    Extracts a 1-month history of assets from the Colombian Stock Exchange (BVC) using the Yahoo Finance API.
    Replaces Playwright's headless extraction while maintaining an immutable schema.
    """
    logging.info(f"Starting extraction of the 15 BVC assets through Yahoo Finance...")

    for ticker in tickers_co:
        try:
            # Download the monthly history (daily candles)
            df = yf.download(ticker, period="1mo", interval="1d")

            if df.empty:
                logging.warning(f"⚠ The Colombian asset {ticker} No data was returned in Yahoo Finance.")
                continue

            # Convert to raw CSV for the immutable layer (Bronze Layer)
            raw_csv = df.to_csv()
            filename = f"{ticker}_bvc_raw.csv"
            source_url = f"https://finance.yahoo.com/quote/{ticker}"

            # We save in the 'bvc' folder for the relational model and the agent
            # Please be aware that, contractually, this data represents the local market.
            f_path, _ = writer.save_raw_data(raw_csv, "bvc", filename, source_url)
            logging.info(f"✓ Locally Ingested (Yahoo): {ticker} -> saved in {os.path.basename(f_path)}")

        except Exception as e:
            logging.error(f"✕ Failure to extract local asset {ticker} from Yahoo: {str(e)}")

**2.5 End-to-End Pipeline Orchestrator**

In [11]:
def main_sprint_2_unified_pipeline():
    logging.info("=========================================================")
    logging.info("INITIATING UNIFIED PIPELINE ORCHESTRATION (YAHOO FINANCE)")
    logging.info("=========================================================")

   # 1. Definition of Tickers under the Yahoo Finance standard
    YAHOO_TICKERS_INT = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META", "BABA", "NFLX", "AMD", "V", "MA", "DIS", "JPM", "COIN"]

    # The suffix '.CO' is added, which is essential for Yahoo to locate the BVC shares.
    BVC_TICKERS_YAHOO = [
        "ECOPETROL.CL", "CIBEST.CL", "GRUPOAVAL.CL", "ISA.CL",
        "GRUPOARGOS.CL", "PFCEMARGOS.CL", "NUTRESA.CL", "CEMARGOS.CL",
        "GRUPOSURA.CL", "CORFICOLCF.CL", "BOGOTA.CL", "CELSIA.CL",
        "ETB.CL", "MINEROS.CL", "CNEC.CL"
    ]

   # Recover the Data Lake route (Sprint 1)
    # BASE_PATH is expected to be defined globally by a previous cell.
    # The previous conditional assignment caused an UnboundLocalError due to Python's scoping rules.
    writer = RawDataLakeWriter(BASE_PATH)

    # 2. Execution of the International Intake (Logic of Step 3)
    extract_yahoo_finance(YAHOO_TICKERS_INT, writer)

    # 3. Executing Local Ingest via Yahoo (New Step 4)
    extract_colombian_assets_via_yahoo(BVC_TICKERS_YAHOO, writer)

    logging.info("=========================================================")
    logging.info("=== SPRINT 2 COMPLETED: UNIFIED AND SECURE PIPELINE ===")
    logging.info("=========================================================")

# Direct execution (No longer requires 'await' because we removed Playwright)
main_sprint_2_unified_pipeline()

/tmp/ipykernel_1187/2534518634.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1mo", interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1187/2534518634.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1mo", interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1187/2534518634.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1mo", interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1187/2534518634.py:11: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker, period="1mo", interval="1d")
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_1187/2534518634.py:11

# **3. Processing, Normalization and Relational Model in DuckDB (Silver Layer)**

The goal here is to take the structured/semi-structured files from the Raw Data Lake (Bronze layer) and consolidate them efficiently, in a typed and idempotent manner, into an embedded analytical relational database.

Build the Silver Layer of the Medallion Architecture. Design and implement a distributed/local processing component with DuckDB that reads raw storage, normalizes schemas, removes duplicates (idempotency), and prepares the fact and dimension tables that the AI ​​Agent will query to make investment decisions.

**3.1 Definition of the Database Manager (DuckDB Manager) and Structured Table Design (Relational Model / Silver Layer)**

In [12]:
import duckdb
import os
import logging

class DuckDBManager:
    def __init__(self, base_path: str):
        self.base_path = base_path
        self.db_path = os.path.join(base_path, "db", "trader_platform.db")

        # Ensure the existence of the database directory
        os.makedirs(os.path.dirname(self.db_path), exist_ok=True)
        self.conn = None

    def connect(self):
        """Establishes a persistent connection with the DuckDB file."""
        try:
            self.conn = duckdb.connect(self.db_path)
            logging.info(f"Conexión exitosa a DuckDB en: {self.db_path}")
        except Exception as e:
            logging.error(f"Error connecting to DuckDB: {str(e)}")
            raise e

    def disconnect(self):
        """Close the connection securely."""
        if self.conn:
            self.conn.close()
            logging.info("Conexión a DuckDB cerrada.")

    def execute_query(self, query: str, params=None):
        """Execute a control DDL or DML query."""
        if not self.conn:
            self.connect()
        if params:
            self.conn.execute(query, params)
        else:
            self.conn.execute(query)

    # The method is now correctly indented within the DuckDBManager class
    def initialize_silver_schema(self):
        """Crea el esquema relacional de la Capa Silver si no existe."""
        logging.info("Initializing relational schema (Capa Silver)...")

        # 1. Asset Dimension Table (Market Standardization)
        create_dim_assets = """
        CREATE TABLE IF NOT EXISTS dim_assets (
            ticker VARCHAR PRIMARY KEY,
            market VARCHAR,      -- 'INTERNATIONAL' o 'LOCAL_BVC'
            currency VARCHAR,    -- 'USD' o 'COP'
            last_updated_utc TIMESTAMP
        );
        """

        # 2. Table of Historical Price Facts (Daily Granularity)
        create_fact_prices = """
        CREATE TABLE IF NOT EXISTS fact_historical_prices (
            ticker VARCHAR,
            date DATE,
            open DOUBLE,
            high DOUBLE,
            low DOUBLE,
            close DOUBLE,
            adj_close DOUBLE,
            volume BIGINT,
            ingestion_timestamp TIMESTAMP,
            PRIMARY KEY (ticker, date),
            FOREIGN KEY (ticker) REFERENCES dim_assets(ticker)
        );
        """

        self.execute_query(create_dim_assets)
        self.execute_query(create_fact_prices)
        logging.info("Esquema relacional Silver validado/creado correctamente.")

**3.2 Incremental Processor and Ingestor Development (ETL)**

In [13]:
import pandas as pd
from datetime import datetime, timezone # Import timezone
import logging

class SilverDataProcessor:
    def __init__(self, db_manager: DuckDBManager):
        self.db_manager = db_manager

    def process_and_load_assets(self, tickers: list, market_type: str, currency: str):
        """Register or update asset metadata in dim_assets."""
        logging.info(f"Processing dimensions for {len(tickers)} market assets {market_type}...")

        query = """
        INSERT INTO dim_assets (ticker, market, currency, last_updated_utc)
        VALUES (?, ?, ?, ?)
        ON CONFLICT (ticker) DO UPDATE SET
            market = EXCLUDED.market,
            currency = EXCLUDED.currency,
            last_updated_utc = EXCLUDED.last_updated_utc;
        """
        now = datetime.now(timezone.utc) # Updated to use datetime.now(timezone.utc)
        for ticker in tickers:
            self.db_manager.execute_query(query, (ticker, market_type, currency, now))

    def ingest_prices_dataframe(self, ticker: str, df: pd.DataFrame):
        """Normalize and load a DataFrame of historical prices to fact_historical_prices."""
        if df.empty:
            logging.warning(f"Empty DataFrame received for {ticker}. Skipping transactions.")
            return

        # Standardize column names in Pandas DataFrames
        df = df.reset_index()
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]

        # Ensure required columns in the fact table
        df['ticker'] = ticker
        df['ingestion_timestamp'] = datetime.now(timezone.utc) # Updated to use datetime.now(timezone.utc)

        # Explicit mapping to the DuckDB structure
        required_cols = ['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ingestion_timestamp']

        # If 'adj_close' does not exist (for example, in the synthetic fallback), it is set to 'close'
        if 'adj_close' not in df.columns:
            df['adj_close'] = df['close']

        final_df = df[required_cols].copy()
        final_df['date'] = pd.to_datetime(final_df['date']).dt.date

        # Direct idempotent insertion using DuckDB's native integration with Pandas
        # We temporarily register the local DataFrame in the DuckDB session
        con = self.db_manager.conn

        # Avoid duplicates by first deleting the temporary window that will be rewritten (Delete-and-Insert strategy)
        min_date = final_df['date'].min()
        max_date = final_df['date'].max()

        con.execute(
            "DELETE FROM fact_historical_prices WHERE ticker = ? AND date BETWEEN ? AND ?",
            (ticker, min_date, max_date)
        )

        # Insert clean records
        con.execute("INSERT INTO fact_historical_prices SELECT * FROM final_df")
        logging.info(f"Silver Layer: {len(final_df)} records successfully indexed for {ticker}.")

**3.4 Unified Orchestrator**

In [14]:
def main_sprint_3_database_and_processing_pipeline():
    logging.info("=========================================================")
    logging.info("INITIATING SILVER LAYER RELATIONAL ORCHESTRATION")
    logging.info("=========================================================")

    # 1. Definition of Tickers
    YAHOO_TICKERS_INT = ["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META", "BABA", "NFLX", "AMD", "V", "MA", "DIS", "JPM", "COIN"]
    BVC_TICKERS_YAHOO = [
        "ECOPETROL.CL", "CIBEST.CL", "GRUPOAVAL.CL", "ISA.CL",
        "GRUPOARGOS.CL", "PFCEMARGOS.CL", "NUTRESA.CL", "CEMARGOS.CL",
        "GRUPOSURA.CL", "CORFICOLCF.CL", "BOGOTA.CL", "CELSIA.CL",
        "ETB.CL", "MINEROS.CL", "CNEC.CL", "PFGRUPOARG.CL" # Added PFGRUPOARG.CL
    ]

    # 2. Initialize DuckDB Infrastructure (Retrieves global BASE_PATH)
    db_manager = DuckDBManager(BASE_PATH)
    db_manager.connect()
    db_manager.initialize_silver_schema()

    # 3. Initialize Silver Processor
    processor = SilverDataProcessor(db_manager)

    try:
        # 4. Register Asset Dimensions
        processor.process_and_load_assets(YAHOO_TICKERS_INT, market_type="INTERNATIONAL", currency="USD")
        processor.process_and_load_assets(BVC_TICKERS_YAHOO, market_type="LOCAL_BVC", currency="COP")

        # 5. Ingesting from the Bronze Layer to the Silver Layer (Incremental Processing Simulation)
        # Note: Here I assume that your functions from the previous sprint returned the data or read it from the hard drive.
        # If your functions saved .csv files to the Raw Data Lake, DuckDB reads them directly from the path using:
        # con.execute("INSERT INTO fact_historical_prices SELECT ... FROM read_csv_auto('path_sprint_1')")

        logging.info("Running validation queries on the database...")
        # Control Validation: Count how many assets there are per market
        df_summary = db_manager.conn.execute("""
            SELECT market, count(*), currency
            FROM dim_assets
            GROUP BY market, currency
        """).fetchdf()

        print("\n=== DIM_ASSETS SUMMARY INGESTEDS ===")
        print(df_summary.to_string(index=False))
        print("========================================\n")

    except Exception as e:
        logging.error(f"Critical failure in the execution of Sprint 3: {str(e)}")

    finally:
        # Always close the connection securely
        db_manager.disconnect()

    logging.info("=========================================================")
    logging.info("=== SPRINT 3 COMPLETED: DUCKDB PROCESSED LAYER READY ===")
    logging.info("=========================================================")

# Direct execution in the Colab cell
main_sprint_3_database_and_processing_pipeline()


=== DIM_ASSETS SUMMARY INGESTEDS ===
       market  count_star() currency
INTERNATIONAL            15      USD
    LOCAL_BVC            16      COP



**3.5 Initiating massive incremental ingestion into the fact table**

In [15]:
import os
import pandas as pd
import duckdb
from datetime import datetime, timezone

# 1. Reconnect to the DuckDB database
db_manager = DuckDBManager(BASE_PATH)
db_manager.connect()

# 2. Bronze Layer Route
bronze_dir = "/content/drive/MyDrive/Trading_Agent_POCv1/data/raw"

print("🚀 Initiating massive incremental ingestion into the fact table (Layer Silver)...")
print(f"📂 Searching for historical archives in: {bronze_dir}")

total_records = 0

if os.path.exists(bronze_dir):
# Search through the Data Lake subfolders looking for pricing files
    for root, dirs, files in os.walk(bronze_dir):
        for file in files:
            # Detect pricing files (.csv)
            if file.endswith("_prices.csv") or file.endswith(".csv") or "_raw.csv" in file:
                # Extract the name of the clean ticker based on the file name
                ticker = file.split('_')[0].upper()
                file_path = os.path.join(root, file)

                try:
                    # We load the CSV ensuring that it handles the first column (the dates) correctly
                    df_raw = pd.read_csv(file_path)

                    if df_raw.empty:
                        continue

                    # KEY NOTE: Since yfinance stores the date in the first column without a header name,
                    # We convert the first physical column (index 0) into our official 'date' column
                    original_date_column = df_raw.columns[0]
                    df_raw = df_raw.rename(columns={original_date_column: 'date'})

                    # Convert the rest of the columns to lowercase and clear spaces
                    df_raw.columns = [c.lower().replace(" ", "_") for c in df_raw.columns]

                    # Ensure the existence of the required columns for the Silver scheme
                    df_raw['ticker'] = ticker
                    df_raw['ingestion_timestamp'] = datetime.now(timezone.utc)

                    # If adj_close does not exist (very common), we set it equal to close
                    if 'adj_close' not in df_raw.columns:
                        if 'close' in df_raw.columns:
                            df_raw['adj_close'] = df_raw['close']
                        else:
                            df_raw['adj_close'] = 0.0

                    # Sort and filter only the columns valid for the final table
                    columnas_destino = ['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'ingestion_timestamp']
                    df_final = df_raw[[c for c in columnas_destino if c in df_raw.columns]].copy()

                    # Clean and format dates to standard YYYY-MM-DD strings to avoid errors in DuckDB
                    df_final['date'] = pd.to_datetime(df_final['date'], errors='coerce', format='%Y-%m-%d').dt.strftime('%Y-%m-%d')

                    # Remove rows where the date could not be parsed correctly
                    df_final = df_final.dropna(subset=['date'])

                    if df_final.empty:
                        continue

                    # Clean and idempotent bulk dump directly to DuckDB
                    con = db_manager.conn
                    con.execute("""
                        INSERT OR REPLACE INTO fact_historical_prices (
                            ticker, date, open, high, low, close, adj_close, volume, ingestion_timestamp
                        )
                        SELECT
                            ticker,
                            CAST(date AS DATE),
                            CAST(open AS DOUBLE),
                            CAST(high AS DOUBLE),
                            CAST(low AS DOUBLE),
                            CAST(close AS DOUBLE),
                            CAST(adj_close AS DOUBLE),
                            CAST(volume AS BIGINT),
                            CAST(ingestion_timestamp AS TIMESTAMP)
                        FROM df_final
                    """)

                    count = len(df_final)
                    total_records += count
                    print(f"  ✅ Successfully ingested {count} records for the asset: {ticker}")

                except Exception as e:
                    print(f"  ❌ Error processing file {file}: {str(e)}")
else:
    print(f"❌ ERROR: The specified path could not be found: {bronze_dir}")

# 3. Resumen y auditoría final
print("\n==============================================")
print(f"🎉 ¡SILVER LAYER PROCESSING COMPLETED!")
print(f"Processed route: {bronze_dir}")
print(f"Total number of indexed price records: {total_records}")
print("==============================================")

# Securely close connection
db_manager.disconnect()

🚀 Initiating massive incremental ingestion into the fact table (Layer Silver)...
📂 Searching for historical archives in: /content/drive/MyDrive/Trading_Agent_POCv1/data/raw
  ✅ Successfully ingested 22 records for the asset: ISA.CL
  ✅ Successfully ingested 22 records for the asset: GRUPOARGOS.CL
  ✅ Successfully ingested 22 records for the asset: PFGRUPOARG.CL
  ✅ Successfully ingested 22 records for the asset: NUTRESA.CL
  ✅ Successfully ingested 22 records for the asset: CEMARGOS.CL
  ✅ Successfully ingested 22 records for the asset: BOGOTA.CL
  ✅ Successfully ingested 22 records for the asset: CELSIA.CL
  ✅ Successfully ingested 22 records for the asset: ETB.CL
  ✅ Successfully ingested 22 records for the asset: MINEROS.CL
  ✅ Successfully ingested 22 records for the asset: CNEC.CL
  ✅ Successfully ingested 22 records for the asset: ECOPETROL.CL
  ✅ Successfully ingested 19 records for the asset: GRUPOAVAL.CL
  ✅ Successfully ingested 19 records for the asset: GRUPOSURA.CL
  ✅ Succ

**3.6 quick audit consultation**

In [16]:
# Initialize and connect for a quick audit query
db_manager = DuckDBManager(BASE_PATH)
db_manager.connect()

# View the first 18 rows of saved historical prices
df_audit_prices = db_manager.conn.execute("""
    SELECT ticker, date, open, high, low, close, volume, ingestion_timestamp
    FROM fact_historical_prices
    LIMIT 100
""").fetchdf()

# Disconnect safely
db_manager.disconnect()

# Mostrar la tabla formateada en Colab
print("\n=== SAMPLE_FACT_HISTORICAL_PRICES) ===")
display(df_audit_prices)


=== SAMPLE_FACT_HISTORICAL_PRICES) ===


,ticker,date,open,high,low,close,volume,ingestion_timestamp
0,ISA.CL,2026-04-15,30480.000000,30800.000000,29900.000000,30740.000000,228379,2026-06-09 01:03:20.905296
1,ISA.CL,2026-04-16,30740.000000,30800.000000,30080.000000,30480.000000,175558,2026-06-09 01:03:20.905296
2,ISA.CL,2026-04-17,30480.000000,30800.000000,30180.000000,30180.000000,156217,2026-06-09 01:03:20.905296
3,ISA.CL,2026-04-20,30180.000000,30300.000000,28980.000000,28980.000000,234349,2026-06-09 01:03:45.837948
4,ISA.CL,2026-04-21,28757.547307,29650.638838,28380.464216,28896.472656,81343,2026-06-09 01:03:54.427411
...,...,...,...,...,...,...,...,...
95,BOGOTA.CL,2026-04-24,38442.917053,39318.882020,38263.742401,38642.000000,4124,2026-06-09 01:04:06.320122
96,BOGOTA.CL,2026-04-27,38628.797687,38509.388730,38011.851408,38190.964844,8694,2026-06-09 01:04:22.827365
97,BOGOTA.CL,2026-04-28,38190.965989,38449.685404,37315.300276,37394.906250,10745,2026-06-09 01:04:31.968276
98,BOGOTA.CL,2026-04-29,37394.907949,37812.839331,37414.809443,37434.710938,13161,2026-06-09 01:04:54.824060


# **4. MULTI-AGENT AI (MULTI-AGENT TRADING)**

The main objective is to build the AI ​​Agent (Agent Trader). This component will interact with the structured DuckDB database (Silver Layer) that you just populated, analyze historical price trends, and generate trading recommendations (Buy, Sell, or Hold) using a Language Model (LLM).

**4.1.1 Installing Dependencies and Configuring the LLM-Gemeni**

In [17]:
!pip install google-genai

In [18]:
import os
import duckdb
import pandas as pd
from datetime import datetime, timezone
from google import genai
from google.genai import types  # Requerido para configuraciones avanzadas
from google.colab import userdata

In [19]:
from google import genai
from google.colab import userdata

try:
    # Inicializar el cliente moderno con tu API Key de los secretos de Colab
    client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

    # Probamos con el modelo actualizado gemini-2.5-flash
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents="Respond only with the word: Successful Connection!"
    )
    print(response.text.strip())
except Exception as e:
    print(f"❌ Configuration error: {str(e)}")

Successful Connection!


**4.1.2 Installing Dependencies and Configuring the LLM-GitHub Models (Copilot Engine)**

In [20]:
# 1. We installed the official OpenAI client compatible with the GitHub endpoint
!pip install -q openai

import openai
from google.colab import userdata

try:
    # We securely recovered the Colab Secrets token
    github_token = userdata.get('GITHUB_TOKEN')

    # We initialize the client by pointing to GitHub's inference infrastructure
    copilot_client = openai.OpenAI(
        base_url="https://models.inference.ai.azure.com",
        api_key=github_token
    )

    # We performed a quick connectivity test with GPT-4o
    print("⏳ Connecting with GitHub Models (Copilot Engine)...")
    test_response = copilot_client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a concise financial assistant."},
            {"role": "user", "content": "Hello, please confirm if you have access to the system."}
        ],
        temperature=0.2,
        max_tokens=50
    )

    print("\n✅ GitHub Models configured correctly!")
    print(f"Model response: {test_response.choices[0].message.content}")

except userdata.NotebookAccessError:
    print("❌ ERROR: No has activado el interruptor de acceso (Notebook access) para 'GITHUB_TOKEN' en el panel de la llave 🔑.")
except Exception as e:
    print(f"❌ Error al conectar con GitHub Models: {e}")

⏳ Connecting with GitHub Models (Copilot Engine)...

✅ GitHub Models configured correctly!
Model response: Hello! I don’t have access to any external systems or personal data. How can I assist you?


**4.1.3 Installing Dependencies and Configuring the LLM-Hugging Face Serverless-Llama-3**

In [21]:
# 1. Instalamos/actualizamos el cliente oficial de Hugging Face
!pip install -q huggingface_hub

from huggingface_hub import InferenceClient
from google.colab import userdata

try:
    # Recuperamos el token de los Secretos de Colab de forma segura
    hf_token = userdata.get('HF_TOKEN')

    # Inicializamos el cliente apuntando al modelo Llama-3 de 70B
    hf_client = InferenceClient(
        model="meta-llama/Meta-Llama-3-70B-Instruct",
        token=hf_token
    )

    print("⏳ Conectando con Hugging Face Serverless Infrastructure (Task: Conversational)...")

    # Construimos la estructura de mensajes nativa compatible con la tarea de chat
    messages = [
        {"role": "system", "content": "Eres un asistente financiero conciso."},
        {"role": "user", "content": "Hola, confirma conexión exitosa a la API."}
    ]

    # Invocamos la API usando chat_completion en lugar de text_generation
    hf_response = hf_client.chat_completion(
        messages=messages,
        max_tokens=50,
        temperature=0.2
    )

    print("\n✅ Hugging Face Serverless configurado y conectado correctamente!")
    print(f"Respuesta del modelo: {hf_response.choices[0].message.content.strip()}")

except userdata.NotebookAccessError:
    print("❌ ERROR: No has activado el interruptor de acceso (Notebook access) para 'HF_TOKEN' en el panel de la llave 🔑.")
except Exception as e:
    print(f"❌ Error al conectar con Hugging Face: {e}")

⏳ Conectando con Hugging Face Serverless Infrastructure (Task: Conversational)...

✅ Hugging Face Serverless configurado y conectado correctamente!
Respuesta del modelo: Conectado con éxito a la API. Listo para procesar solicitudes financieras. ¿En qué puedo ayudarte hoy?


**4.1.4 Installing Dependencies and Configuring the LLM-Hugging Face Serverless-Model Qwen-2.5**

In [22]:
# 1. Instalamos/actualizamos el cliente oficial de Hugging Face
!pip install -q huggingface_hub

from huggingface_hub import InferenceClient
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')

    # CAMBIO AQUÍ: Usamos Qwen 2.5 de 72 Billones de parámetros (Nativo y Libre)
    hf_client = InferenceClient(
        model="Qwen/Qwen2.5-72B-Instruct",
        token=hf_token
    )

    print("⏳ Conectando con Hugging Face Serverless Infrastructure (Model: Qwen 2.5)...")

    messages = [
        {"role": "system", "content": "Eres un asistente financiero conciso."},
        {"role": "user", "content": "Hola, confirma conexión exitosa a la API."}
    ]

    hf_response = hf_client.chat_completion(
        messages=messages,
        max_tokens=50,
        temperature=0.2
    )

    print("\n✅ Hugging Face Serverless configurado y conectado correctamente!")
    print(f"Respuesta del modelo: {hf_response.choices[0].message.content.strip()}")

except userdata.NotebookAccessError:
    print("❌ ERROR: No has activado el interruptor de acceso (Notebook access) para 'HF_TOKEN' en el panel de la llave 🔑.")
except Exception as e:
    print(f"❌ Error al conectar con Hugging Face: {e}")

⏳ Conectando con Hugging Face Serverless Infrastructure (Model: Qwen 2.5)...

✅ Hugging Face Serverless configurado y conectado correctamente!
Respuesta del modelo: No puedo conectarme directamente a APIs ni realizar llamadas a servicios web. Soy un asistente de IA que responde preguntas y genera texto, pero no tengo capacidades de conexión a redes o APIs. Si necesitas confirmar una


**4.2 Creation of Database Tools (Tools)**

In [23]:
def query_trader_database(sql_query: str) -> str:
    """
    Exclusive tool for the AI ​​Agent. Executes read-only queries
    on DuckDB's Silver Layer to analyze historical asset prices.
    """
    db_path = os.path.join(BASE_PATH, "db", "trader_platform.db")
    conn = duckdb.connect(db_path, read_only=True)
    try:
        df = conn.execute(sql_query).fetchdf()
        #The SDK prefers to receive clean strings or JSON from the functions
        return df.to_json(orient="records")
    except Exception as e:
        return f"Error executing SQL: {str(e)}"
    finally:
        conn.close()

**4.3 Definition of the Reasoning System (Prompt Engineering)**

In [24]:
TRADER_AGENT_PROMPT = """
You are an AI Agent Expert in Quantitative Trading and Financial Market Analyst.
Your goal is to analyze the historical price data in the 'fact_historical_prices' table to issue rigorous operational recommendations.

OPERATING RULES:
1. Always use the 'query_trader_database' tool to query the necessary data.
2. Analyze short-term trends by evaluating opening and closing prices, volume, and volatility.
3. Your decisions must be based strictly on the numerical data returned by the database.
4. When analyzing volatility, consider the range of prices (high-low) and changes in closing prices over the period.
5. Include a description of a line chart that visualizes the asset's price and volatility trends.

You must ALWAYS respond in JSON format with the following structure.:
{
  "ticker": "TICKER_NAME",
  "trend_analysis": "Brief qualitative explanation of what was observed, incorporating price, volume, and volatility.",
  "volatility_analysis": "Explanation of volatility trends and its impact on the recommendation.",
  "volatility_metric": "Calculated volatility value (e.g., average daily range or standard deviation if calculable from the provided data) for the period.",
  "key_metric": "Current price or significant variations",
  "recommendation": "BUY" | "SELL" | "HOLD",
  "technical_justification": "Numerical support for why you made the decision, referencing price movements, volume, and volatility.",
  "chart_suggestion": "Describe a line chart showing closing prices over time, highlighting periods of high/low volatility and trend lines. Explain what this chart would visually convey."
}
"""

**4.4 Definition of the Reasoning System (technical and fundamental analysis agent)**

In [25]:
ADVANCED_TRADER_AGENT_PROMPT = """
You are an AI Agent specializing in quantitative trading and financial market analysis, with a focus on advanced technical indicators.
Your objective is to provide an in-depth analytical report on an asset, incorporating multiple available technical indicators from historical price data and suggesting detailed chart visualizations.

OPERATING RULES:
1. Always use the 'query_trader_database' tool to query historical price data (date, open, high, low, close, volume).
2. Retrieve the last 50 records from the 'fact_historical_prices' table to allow for the calculation of longer-term indicators.
3. Identify and analyze at least three key technical indicators (e.g., Relative Strength Index (RSI), Simple Moving Average (SMA), and describe how Volume could be used for analysis).
4. Provide a detailed analysis based on these indicators, their trends, and potential signals (e.g., RSI overbought/oversold crossovers, SMA crossovers, volume analysis).
5. Describe a combined line chart that visualizes closing prices, the identified technical indicators, and their relationships.

You must always respond in JSON format with the following structure:
{
"ticker": "ASSET_NAME",
"analysis_period_days": 50,
"technical_indicators_suggested": ["RSI_15", "SMA_20", "Volume_Analysis"], // Example indicators you could consider
"technical_analysis_report": "Detailed report based on the identified indicators, including trends, signals, and implications. You can mention volume analysis here."
"chart_suggestion": "Describe a line chart showing closing prices, along with SMA lines. If possible, mention how volume could be incorporated into a subchart. Explain how these elements interact visually to support the analysis."
"recommendation": "BUY" | "SELL" | "HOLD",
"justification": "Based on technical indicators and trends."
}
"""

**4.5 Definition of the Reasoning System (Portfolio Manager Agent)**

In [26]:
PORTFOLIO_MANAGER_AGENT_PROMPT = """
You are an AI Agent specializing in Portfolio Management.
Your main objective is to synthesize the recommendations and analyses from the 'TRADER_AGENT_PROMPT' and 'ADVANCED_TRADER_AGENT_PROMPT' to make a final, consolidated investment decision (BUY, SELL, or HOLD) for a given asset.

OPERATING RULES:
1. You will receive the output (in JSON format) from two other agents:
    - TRADER_AGENT_PROMPT (basic technical analysis).
    - ADVANCED_TRADER_AGENT_PROMPT (advanced technical indicators analysis).
2. Carefully evaluate both recommendations and their justifications.
3. Identify consensus or discrepancies between the two agents' opinions.
4. If there's a strong consensus, reinforce that decision.
5. If there are discrepancies, weigh the strengths of the justifications and the depth of the analysis to arrive at a final decision. The ADVANCED_TRADER_AGENT_PROMPT's analysis should generally carry more weight due to its deeper technical insight, but consider the TRADER_AGENT_PROMPT's focus on recent behavior and recent sentiment.
6. Provide a clear, final recommendation along with a comprehensive justification based on the combined inputs.

You must ALWAYS respond in JSON format with the following structure:
{
  "ticker": "TICKER_NAME",
  "trader_agent_recommendation": "Recommendation from TRADER_AGENT_PROMPT",
  "trader_agent_justification": "Justification from TRADER_AGENT_PROMPT",
  "advanced_trader_agent_recommendation": "Recommendation from ADVANCED_TRADER_AGENT_PROMPT",
  "advanced_trader_agent_justification": "Justification from ADVANCED_TRADER_AGENT_PROMPT",
  "final_recommendation": "BUY" | "SELL" | "HOLD",
  "portfolio_manager_justification": "Detailed justification for the final decision, explaining how both agents' inputs were weighed and consolidated."
}
"""

**4.6 Agent Implementation and Decision Orchestration**

In [27]:
#@title ⚙️ Execution Control { display-mode: "form" }
execute_agents = False #@param {type:"boolean"}

if execute_agents:
    # -------------------------------------------------------------
    # PEGA AQUÍ TODO TU CÓDIGO ORIGINAL (El bucle, los agentes, etc.)
    # -------------------------------------------------------------
    print("🤖 Ejecutando análisis de agentes...")
    # Tu código actual...

else:
    print("⏭️ Celda omitida automáticamente para cuidar la cuota de la API.")
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO


class AgentTraderPlatform:
    def __init__(self):
       # Cliente unificado del nuevo SDK google-genai
        self.client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        self.model_name = "gemini-2.5-flash"

    def analyze_asset(self, ticker: str) -> str:
        print(f"🤖 Basic AI agent initiating technical analysis for: {ticker}...")

        prompt_usuario = f"""
        Perform an analytical analysis of the asset with ticker '{ticker}'.
        Consult its last 15 price records in the 'fact_historical_prices' table
        to evaluate its recent behavior, including price, volume, and volatility.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """

        try:
            # In the new SDK, system tools and instructions are passed
            # within the GenerateContentConfig object
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_usuario,
                config=types.GenerateContentConfig(
                    system_instruction=TRADER_AGENT_PROMPT,
                    tools=[query_trader_database]
                    # response_mime_type="application/json"  # Removed this line as it's unsupported with tools
                )
            )
            agent_verdict_json_str = response.text

            # --- Start: Integrated Chart Generation ---
            try:
                clean_agent_verdict = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_verdict)

                ticker_to_plot = agent_output.get('ticker', ticker) # Use the ticker from agent output, fallback to input
                # The agent specifies "last 15 records", so we query for that.
                sql_query = f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT 15"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]': # Check if data_json is not empty or an empty list
                    df_plot = pd.read_json(StringIO(data_json))

                    # Ensure date is in datetime format and sort chronologically
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)

                    # Calculate daily range for volatility visualization
                    df_plot['daily_range'] = df_plot['high'] - df_plot['low']

                    # Create the plot as suggested by the agent
                    plt.figure(figsize=(14, 7))
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price')

                    # Add shaded area for volatility (daily high-low range)
                    plt.fill_between(df_plot['date'], df_plot['low'], df_plot['high'], color='skyblue', alpha=0.3, label='Daily Price Range (Volatility)')

                    plt.title(f'{ticker_to_plot} Closing Prices and Daily Volatility (Last 15 Days)', fontsize=16)
                    plt.xlabel('Date', fontsize=12)
                    plt.ylabel('Price', fontsize=12)
                    plt.xticks(rotation=45)
                    plt.grid(True, linestyle='--', alpha=0.7)
                    plt.legend()
                    plt.tight_layout()
                    plt.show()
                    print(f"📈 Chart generated for {ticker_to_plot}.")
                else:
                    print(f"⚠️ No historical data available to plot for {ticker_to_plot}.")
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse agent's JSON output for plotting: {clean_agent_verdict}")
            except Exception as plot_e:
                print(f"❌ Error during chart generation: {plot_e}")
            # --- End: Integrated Chart Generation ---

            return agent_verdict_json_str
        except Exception as e:
            # If agent execution itself fails
            return f"❌ Error in agent execution: {str(e)}"

    def analyze_advanced_indicators(self, ticker: str) -> str:
        print(f"🤖 Advanced AI agent initiating advanced indicator analysis for basic: {ticker}...")

        prompt_usuario_advanced = f"""
        Perform a detailed analysis of the asset with ticker '{ticker}'.
        Consult its last 50 price records in the 'fact_historical_prices' table
        to evaluate its performance using technical indicators such as the RSI, SMA, and volume.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_usuario_advanced,
                config=types.GenerateContentConfig(
                    system_instruction=ADVANCED_TRADER_AGENT_PROMPT,
                    tools=[query_trader_database]
                )
            )
            agent_output_json_str = response.text
            clean_agent_output = agent_output_json_str.replace('```json', '').replace('```', '').strip()
            agent_output = json.loads(clean_agent_output)

            ticker_to_plot = agent_output.get('ticker', ticker)
            analysis_period = agent_output.get('analysis_period_days', 50)

            # Consult data to graph and calculate indicators
            sql_query = f"SELECT date, open, high, low, close, volume FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT {analysis_period}"
            data_json = query_trader_database(sql_query)

            if data_json and data_json != '[]':
                df_plot = pd.read_json(StringIO(data_json))
                df_plot['date'] = pd.to_datetime(df_plot['date'])
                df_plot = df_plot.sort_values('date').reset_index(drop=True)

                # --- Home: Generation of Technical Indicators ---
                # Example: Calculate Simple Moving Average (SMA) for 20 periods
                if not df_plot.empty and len(df_plot) >= 20: # Ensure you have enough data for SMA_20
                    df_plot['SMA_20'] = df_plot['close'].rolling(window=20).mean()
                # For RSI,
                if not df_plot.empty and len(df_plot) >= 14:
                  delta = df_plot['close'].diff()
                  gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()

                  rs = gain / loss
                  df_plot['rsi_14'] = 100 - (100 / (1 + rs))
                # --- End: Generation of Technical Indicators ---

                # --- Home: Generating Charts for Advanced Indicators ---
                plt.figure(figsize=(16, 9))

                # Subchart 1: Price and SMA
                ax1 = plt.subplot(2, 1, 1) # 2 rows, 1 column, first chart
                sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price', ax=ax1)
                if 'SMA_20' in df_plot.columns:
                    sns.lineplot(x='date', y='SMA_20', data=df_plot, color='red', label='SMA 20', linestyle='--', ax=ax1)
                ax1.set_title(f'Prices and Technical Indicators of {ticker_to_plot} (Latest {analysis_period} Days)', fontsize=16)
                ax1.set_ylabel('Price', fontsize=12)
                ax1.legend()
                ax1.grid(True, linestyle='--', alpha=0.7)
                plt.setp(ax1.get_xticklabels(), visible=False) # Hide X-axis labels for the chart above

                # Placeholder for RSI or other indicator on a separate subchart
                ax2 = plt.subplot(2, 1, 2, sharex=ax1) # Share X-axis with the first graph
                # If RSI were calculated: sns.lineplot(x='date', y='RSI', data=df_plot, color='green', label='RSI', ax=ax2)
                ax2.axhline(70, linestyle='--', alpha=0.5, color='red')
                ax2.axhline(30, linestyle='--', alpha=0.5, color='green')
                ax2.set_title('Placeholder for RSI or other Momentum Indicator', fontsize=14)
                ax2.set_ylabel('Indicator Value', fontsize=12)
                ax2.set_xlabel('Date', fontsize=12)
                ax2.grid(True, linestyle='--', alpha=0.7)
                ax2.legend(['Overbought (70)', 'Oversold (30)']) # Placeholder Legend
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

                print(f"📈 Advanced Indicators Chart generated for {ticker_to_plot}.")
                # --- End: Generating Charts for Advanced Indicators ---
            else:
                print(f"⚠️ No historical data is available for graphing. {ticker_to_plot} for advanced analysis.")

            return agent_output_json_str
        except json.JSONDecodeError:
            print(f"⚠️ The JSON output from the agent could not be analyzed for advanced analysis.: {clean_agent_output}")
        except Exception as e:
            return f"❌ Error in the execution of the advanced analysis agent: {str(e)}"

    def manage_portfolio_decision(self, ticker: str) -> str:
        print(f"🧠 Portfolio Manager Agent synthesizing decisions for: {ticker}...")

        # Get recommendations from sub-agents
        trader_report = self.analyze_asset(ticker)
        advanced_trader_report = self.analyze_advanced_indicators(ticker)

        trader_recommendation = "N/A"
        trader_justification = "N/A"
        advanced_trader_recommendation = "N/A"
        advanced_trader_justification = "N/A"

        try:
            # Try to parse the basic trader report
            clean_trader_report = trader_report.replace('```json', '').replace('```', '').strip()
            trader_output = json.loads(clean_trader_report)
            trader_recommendation = trader_output.get('recommendation', 'N/A')
            trader_justification = trader_output.get('technical_justification', 'N/A')
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse TRADER_AGENT_PROMPT output: {trader_report}")
        except Exception as e:
            print(f"❌ Error processing TRADER_AGENT_PROMPT output: {str(e)}")

        try:
            # Try to parse the advanced trader report
            clean_advanced_trader_report = advanced_trader_report.replace('```json', '').replace('```', '').strip()
            advanced_trader_output = json.loads(clean_advanced_trader_report)
            advanced_trader_recommendation = advanced_trader_output.get('recommendation', 'N/A')
            advanced_trader_justification = advanced_trader_output.get('justification', 'N/A')
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse ADVANCED_TRADER_AGENT_PROMPT output: {advanced_trader_report}")
        except Exception as e:
            print(f"❌ Error processing ADVANCED_TRADER_AGENT_PROMPT output: {str(e)}")

        # Construct the prompt for the Portfolio Manager Agent
        portfolio_manager_prompt_user = f"""
        Analyze the following reports for ticker '{ticker}':

        TRADER_AGENT_PROMPT Report:
        Recommendation: {trader_recommendation}
        Justification: {trader_justification}

        ADVANCED_TRADER_AGENT_PROMPT Report:
        Recommendation: {advanced_trader_recommendation}
        Justification: {advanced_trader_justification}

        Based on these inputs, provide a final consolidated investment decision.
        """

        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=portfolio_manager_prompt_user,
                config=types.GenerateContentConfig(
                    system_instruction=PORTFOLIO_MANAGER_AGENT_PROMPT,
                    # No tools are directly used by the Portfolio Manager as it synthesizes other agents' outputs
                )
            )
            portfolio_decision_json_str = response.text
            clean_portfolio_decision = portfolio_decision_json_str.replace('```json', '').replace('```', '').strip()

            # Print the decision immediately before returning
            print("\n================ PORTFOLIO MANAGER AGENT'S DECISION ================")
            print(clean_portfolio_decision)
            print("=====================================================================")

            return clean_portfolio_decision
        except Exception as e:
            return f"❌ Error in Portfolio Manager Agent execution: {str(e)}"


# =====================================================================
# 4. SPRINT 4 EXECUTION AND TESTING
# =====================================================================
# Instantiate the modern trading agent
agente = AgentTraderPlatform()

# We tested with an existing asset in your DuckDB (e.g., ECOPETROL or AAPL)
ticker_a_probar = "GOOGL" #["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "ECOPETROL.CL"]

agent_report = agente.analyze_asset(ticker_a_probar)

print("\n================AI AGENT'S VERDICT ================")
print(agent_report)
print("=====================================================================")

print("\n================ TESTING NEW ADVANCED ANALYTICS AGENT ================")
advanced_agent_report = agente.analyze_advanced_indicators(ticker_a_probar)
print("\n================ ADVANCED ANALYSIS AGENT'S VERDICT ================")
print(advanced_agent_report)
print("=====================================================================")

# Test the new Portfolio Manager Agent
portfolio_decision = agente.manage_portfolio_decision(ticker_a_probar)

⏭️ Celda omitida automáticamente para cuidar la cuota de la API.
🤖 Basic AI agent initiating technical analysis for: GOOGL...

================AI AGENT'S VERDICT ================
❌ Error in agent execution: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

================ TESTING NEW ADVANCED ANALYTICS AGENT ================
🤖 Advanced AI agent initiating advanced indicator analysis for basic: GOOGL...

================ ADVANCED ANALYSIS AGENT'S VERDICT ================
❌ Error in the execution of the advanced analysis agent: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
🧠 Portfolio Manager Agent synthesizing decisions for: GOOGL...
🤖 Basic AI agent initiating technical analysis for: GOOGL...
🤖 

**4.7 Agent Implementation and Decision Orchestration with Caching (Result Persistence)**

In [28]:
#@title ⚙️ Execution Control { display-mode: "form" }
execute_agents = False #@param {type:"boolean"}

if execute_agents:
    # -------------------------------------------------------------
    # PEGA AQUÍ TODO TU CÓDIGO ORIGINAL (El bucle, los agentes, etc.)
    # -------------------------------------------------------------
    print("🤖 Ejecutando análisis de agentes...")
    # Tu código actual...

else:
    print("⏭️ Celda omitida automáticamente para cuidar la cuota de la API.")



import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import os


class AgentTraderPlatform:
    def __init__(self):
       # Unified client of the new google-genai SDK
        self.client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        self.model_name = "gemini-2.5-flash"
        self._cache = {}
        # Define the cache file path using BASE_PATH (assumed to be global)
        self._cache_file_path = os.path.join(BASE_PATH, 'agent_cache.json')
        self._load_cache() # Load cache on initialization

    def _load_cache(self):
        if os.path.exists(self._cache_file_path):
            try:
                with open(self._cache_file_path, 'r', encoding='utf-8') as f:
                    self._cache = json.load(f)
                print(f"Cache loaded from {self._cache_file_path}")
            except json.JSONDecodeError as e:
                print(f"Error decoding cache file: {e}. Starting with empty cache.")
                self._cache = {}
            except Exception as e:
                print(f"Error loading cache file: {e}. Starting with empty cache.")
                self._cache = {}
        else:
            print("No cache file found. Starting with empty cache.")

    def _save_cache(self):
        try:
            # Ensure the directory exists
            os.makedirs(os.path.dirname(self._cache_file_path), exist_ok=True)
            with open(self._cache_file_path, 'w', encoding='utf-8') as f:
                json.dump(self._cache, f, indent=4)
            print(f"Cache saved to {self._cache_file_path}")
        except Exception as e:
            print(f"Error saving cache file: {e}")

    def analyze_asset(self, ticker: str) -> str:
        print(f"🤖 Basic AI agent initiating technical analysis for: {ticker}...")

        cache_key = f"basic_analysis_{ticker}"
        if cache_key in self._cache:
            print(f"Retrieving basic analysis for {ticker} from cache.")
            return self._cache[cache_key]

        prompt_usuario = f"""
        Perform an analytical analysis of the asset with ticker '{ticker}'.
        Consult its last 15 price records in the 'fact_historical_prices' table
        to evaluate its recent behavior, including price, volume, and volatility.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """

        try:
            # In the new SDK, system tools and instructions are passed
            # within the GenerateContentConfig object
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_usuario,
                config=types.GenerateContentConfig(
                    system_instruction=TRADER_AGENT_PROMPT,
                    tools=[query_trader_database]
                    # response_mime_type="application/json"  # Removed this line as it's unsupported with tools
                )
            )
            agent_verdict_json_str = response.text
            self._cache[cache_key] = agent_verdict_json_str # Cache the result
            self._save_cache() # Save cache after modification

            # --- Start: Integrated Chart Generation ---
            try:
                clean_agent_verdict = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_verdict)

                ticker_to_plot = agent_output.get('ticker', ticker) # Use the ticker from agent output, fallback to input
                # The agent specifies "last 15 records", so we query for that.
                sql_query = f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT 15"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]': # Check if data_json is not empty or an empty list
                    df_plot = pd.read_json(StringIO(data_json))

                    # Ensure date is in datetime format and sort chronologically
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)

                    # Calculate daily range for volatility visualization
                    df_plot['daily_range'] = df_plot['high'] - df_plot['low']

                    # Create the plot as suggested by the agent
                    plt.figure(figsize=(14, 7))
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price')

                    # Add shaded area for volatility (daily high-low range)
                    plt.fill_between(df_plot['date'], df_plot['low'], df_plot['high'], color='skyblue', alpha=0.3, label='Daily Price Range (Volatility)')

                    plt.title(f'{ticker_to_plot} Closing Prices and Daily Volatility (Last 15 Days)', fontsize=16)
                    plt.xlabel('Date', fontsize=12)
                    plt.ylabel('Price', fontsize=12)
                    plt.xticks(rotation=45)
                    plt.grid(True, linestyle='--', alpha=0.7)
                    plt.legend()
                    plt.tight_layout()
                    plt.show()
                    print(f"📈 Chart generated for {ticker_to_plot}.")
                else:
                    print(f"⚠️ No historical data available to plot for {ticker_to_plot}.")
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse agent's JSON output for plotting: {clean_agent_verdict}")
            except Exception as plot_e:
                print(f"❌ Error during chart generation: {plot_e}")
            # --- End: Integrated Chart Generation ---

            return agent_verdict_json_str
        except Exception as e:
            # If agent execution itself fails
            return f"❌ Error in agent execution: {str(e)}"

    def analyze_advanced_indicators(self, ticker: str) -> str:
        print(f"🤖 Advanced AI agent initiating advanced indicator analysis for basic: {ticker}...")

        cache_key = f"advanced_analysis_{ticker}"
        if cache_key in self._cache:
            print(f"Retrieving advanced analysis for {ticker} from cache.")
            return self._cache[cache_key]

        prompt_usuario_advanced = f"""
        Perform a detailed analysis of the asset with ticker '{ticker}'.
        Consult its last 50 price records in the 'fact_historical_prices' table
        to evaluate its performance using technical indicators such as the RSI, SMA, and volume.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_usuario_advanced,
                config=types.GenerateContentConfig(
                    system_instruction=ADVANCED_TRADER_AGENT_PROMPT,
                    tools=[query_trader_database]
                )
            )
            agent_output_json_str = response.text
            self._cache[cache_key] = agent_output_json_str # Cache the result
            self._save_cache() # Save cache after modification

            clean_agent_output = agent_output_json_str.replace('```json', '').replace('```', '').strip()
            agent_output = json.loads(clean_agent_output)

            ticker_to_plot = agent_output.get('ticker', ticker)
            analysis_period = agent_output.get('analysis_period_days', 50)

            # Consult data to graph and calculate indicators
            sql_query = f"SELECT date, open, high, low, close, volume FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT {analysis_period}"
            data_json = query_trader_database(sql_query)

            if data_json and data_json != '[]':
                df_plot = pd.read_json(StringIO(data_json))
                df_plot['date'] = pd.to_datetime(df_plot['date'])
                df_plot = df_plot.sort_values('date').reset_index(drop=True)

                # --- Home: Generation of Technical Indicators ---
                # Example: Calculate Simple Moving Average (SMA) for 20 periods
                if not df_plot.empty and len(df_plot) >= 20: # Ensure you have enough data for SMA_20
                    df_plot['SMA_20'] = df_plot['close'].rolling(window=20).mean()
                # For RSI,
                if not df_plot.empty and len(df_plot) >= 14:
                  delta = df_plot['close'].diff()
                  gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                  loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()

                  rs = gain / loss
                  df_plot['rsi_14'] = 100 - (100 / (1 + rs))
                # --- End: Generation of Technical Indicators ---

                # --- Home: Generating Charts for Advanced Indicators ---
                plt.figure(figsize=(16, 9))

                # Subchart 1: Price and SMA
                ax1 = plt.subplot(2, 1, 1) # 2 rows, 1 column, first chart
                sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price', ax=ax1)
                if 'SMA_20' in df_plot.columns:
                    sns.lineplot(x='date', y='SMA_20', data=df_plot, color='red', label='SMA 20', linestyle='--', ax=ax1)
                ax1.set_title(f'Prices and Technical Indicators of {ticker_to_plot} (Latest {analysis_period} Days)', fontsize=16)
                ax1.set_ylabel('Price', fontsize=12)
                ax1.legend()
                ax1.grid(True, linestyle='--', alpha=0.7)
                plt.setp(ax1.get_xticklabels(), visible=False) # Hide X-axis labels for the chart above

                # Placeholder for RSI or other indicator on a separate subchart
                ax2 = plt.subplot(2, 1, 2, sharex=ax1) # Share X-axis with the first graph
                # If RSI were calculated: sns.lineplot(x='date', y='RSI', data=df_plot, color='green', label='RSI', ax=ax2)
                if 'rsi_14' in df_plot.columns:
                    sns.lineplot(x='date', y='rsi_14', data=df_plot, color='green', label='RSI 14', ax=ax2)
                ax2.axhline(70, linestyle='--', alpha=0.5, color='red')
                ax2.axhline(30, linestyle='--', alpha=0.5, color='green')
                ax2.set_title('RSI (Relative Strength Index)', fontsize=14)
                ax2.set_ylabel('Indicator Value', fontsize=12)
                ax2.set_xlabel('Date', fontsize=12)
                ax2.grid(True, linestyle='--', alpha=0.7)
                ax2.legend(['Overbought (70)', 'Oversold (30)']) # Placeholder Legend
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()

                print(f"📈 Advanced Indicators Chart generated for {ticker_to_plot}.")
                # --- End: Generating Charts for Advanced Indicators ---
            else:
                print(f"⚠️ No historical data is available for graphing. {ticker_to_plot} for advanced analysis.")

            return agent_output_json_str
        except json.JSONDecodeError:
            print(f"⚠️ The JSON output from the agent could not be analyzed for advanced analysis.: {clean_agent_output}")
        except Exception as e:
            return f"❌ Error in the execution of the advanced analysis agent: {str(e)}"

    def manage_portfolio_decision(self, ticker: str) -> str:
        print(f"🧠 Portfolio Manager Agent synthesizing decisions for: {ticker}...")

        # Get recommendations from sub-agents
        trader_report = self.analyze_asset(ticker)
        advanced_trader_report = self.analyze_advanced_indicators(ticker)

        trader_recommendation = "N/A"
        trader_justification = "N/A"
        advanced_trader_recommendation = "N/A"
        advanced_trader_justification = "N/A"

        try:
            # Try to parse the basic trader report
            clean_trader_report = trader_report.replace('```json', '').replace('```', '').strip()
            trader_output = json.loads(clean_trader_report)
            trader_recommendation = trader_output.get('recommendation', 'N/A')
            trader_justification = trader_output.get('technical_justification', 'N/A')
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse TRADER_AGENT_PROMPT output: {trader_report}")
        except Exception as e:
            print(f"❌ Error processing TRADER_AGENT_PROMPT output: {str(e)}")

        try:
            # Try to parse the advanced trader report
            clean_advanced_trader_report = advanced_trader_report.replace('```json', '').replace('```', '').strip()
            advanced_trader_output = json.loads(clean_advanced_trader_report)
            advanced_trader_recommendation = advanced_trader_output.get('recommendation', 'N/A')
            advanced_trader_justification = advanced_trader_output.get('justification', 'N/A')
        except json.JSONDecodeError:
            print(f"⚠️ Could not parse ADVANCED_TRADER_AGENT_PROMPT output: {advanced_trader_report}")
        except Exception as e:
            print(f"❌ Error processing ADVANCED_TRADER_AGENT_PROMPT output: {str(e)}")

        # Construct the prompt for the Portfolio Manager Agent
        portfolio_manager_prompt_user = f"""
        Analyze the following reports for ticker '{ticker}':

        TRADER_AGENT_PROMPT Report:
        Recommendation: {trader_recommendation}
        Justification: {trader_justification}

        ADVANCED_TRADER_AGENT_PROMPT Report:
        Recommendation: {advanced_trader_recommendation}
        Justification: {advanced_trader_justification}

        Based on these inputs, provide a final consolidated investment decision.
        """

        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=portfolio_manager_prompt_user,
                config=types.GenerateContentConfig(
                    system_instruction=PORTFOLIO_MANAGER_AGENT_PROMPT,
                    # No tools are directly used by the Portfolio Manager as it synthesizes other agents' outputs
                )
            )
            portfolio_decision_json_str = response.text
            clean_portfolio_decision = portfolio_decision_json_str.replace('```json', '').replace('```', '').strip()

            # Print the decision immediately before returning
            print("\n================ PORTFOLIO MANAGER AGENT'S DECISION ==================")
            print(clean_portfolio_decision)
            print("======================================================================")

            return clean_portfolio_decision
        except Exception as e:
            return f"❌ Error in Portfolio Manager Agent execution: {str(e)}"


# =====================================================================
# 4. SPRINT 4 EXECUTION AND TESTING
# =====================================================================
# Instantiate the modern trading agent
agente = AgentTraderPlatform()

# We tested with an existing asset in your DuckDB (e.g., ECOPETROL or AAPL)
ticker_a_probar = ["TSLA"]    #["AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "ECOPETROL.CL"]

for ticker_actual in ticker_a_probar:
    print(f"\n\n--- INICIANDO ANÁLISIS PARA EL ACTIVO: {ticker_actual} ---")

    agent_report = agente.analyze_asset(ticker_actual)
    print("\n================AI AGENT'S VERDICT ==================")
    print(agent_report)
    print("======================================================================")

    print("\n================ TESTING NEW ADVANCED ANALYTICS AGENT ==================")
    advanced_agent_report = agente.analyze_advanced_indicators(ticker_actual)
    print("\n================ ADVANCED ANALYSIS AGENT'S VERDICT ==================")
    print(advanced_agent_report)
    print("======================================================================")

    # Test the new Portfolio Manager Agent
    portfolio_decision = agente.manage_portfolio_decision(ticker_actual)



⏭️ Celda omitida automáticamente para cuidar la cuota de la API.
Cache loaded from /content/drive/MyDrive/Trading_Agent_POCv1/agent_cache.json


--- INICIANDO ANÁLISIS PARA EL ACTIVO: TSLA ---
🤖 Basic AI agent initiating technical analysis for: TSLA...
Retrieving basic analysis for TSLA from cache.

================AI AGENT'S VERDICT ==================
```json
{
  "ticker": "TSLA",
  "trend_analysis": "Over the analyzed period, TSLA's price has shown a downward trend from a high of $445.27 to a recent close of $423.74. There is significant volatility, with daily price ranges often exceeding $10. Volume has been relatively high, indicating strong market activity, but the price trend suggests selling pressure.",
  "volatility_analysis": "The volatility is notable, with an average daily range of $13.57 (calculated as the average of daily high-low differences). This indicates significant price fluctuations, which could signal uncertainty or active trading. The downward trend in closing pri

**4.7.a Agent Implementation and Decision Orchestration with Caching (Result Persistence) + RAG control**

****

In [29]:
#@title ⚙️ Execution Control { display-mode: "form" }
execute_agents = True #@param {type:"boolean"}

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import os

class AgentTraderPlatform:
    def __init__(self):
        # Cliente unificado del SDK de google-genai
        self.client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        self.model_name = "gemini-2.5-flash"
        self._cache = {}
        self._cache_file_path = os.path.join(BASE_PATH, 'agent_cache.json')
        self._load_cache()

        # 🔥 LIMPIEZA DE CACHÉ: Forzamos la eliminación de los registros erróneos pasados
        #for key in ["basic_analysis_G", "advanced_analysis_G"]:
        #    if key in self._cache:
        #        del self._cache[key]
        #self._save_cache()

    def _load_cache(self):
        if os.path.exists(self._cache_file_path):
            try:
                with open(self._cache_file_path, 'r', encoding='utf-8') as f:
                    self._cache = json.load(f)
                print(f"Cache loaded from {self._cache_file_path}")
            except Exception as e:
                print(f"Error loading cache file: {e}. Starting with empty cache.")
                self._cache = {}
        else:
            print("No cache file found. Starting with empty cache.")

    def _save_cache(self):
        try:
            os.makedirs(os.path.dirname(self._cache_file_path), exist_ok=True)
            with open(self._cache_file_path, 'w', encoding='utf-8') as f:
                json.dump(self._cache, f, indent=4)
        except Exception as e:
            print(f"Error saving cache file: {e}")

    def analyze_asset(self, ticker: str) -> str:
        print(f"🤖 Basic AI agent initiating technical analysis for: {ticker}...")
        cache_key = f"basic_analysis_{ticker}"
        if cache_key in self._cache:
            print(f"Retrieving basic analysis for {ticker} from cache.")
            return self._cache[cache_key]

        prompt_usuario = f"Perform an analytical analysis of the asset with ticker '{ticker}' consulting its last 15 price records."

        try:
            # --- PASO 1: Llamada inicial al agente ---
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_usuario,
                config=types.GenerateContentConfig(
                    system_instruction=TRADER_AGENT_PROMPT,
                    tools=[query_trader_database]
                )
            )
            primer_output = response.text

            # --- PASO 2: Resolver el bucle de herramientas (Tool/ReAct Loop) ---
            if "query_trader_database" in primer_output and "recommendation" not in primer_output:
                print("🔄 El Agente Básico solicitó datos. Ejecutando query en la base de datos...")
                clean_json = primer_output.replace('```json', '').replace('```', '').strip()
                tool_call = json.loads(clean_json)
                sql_query = tool_call.get("query")

                # Buscamos los datos reales
                datos_reales = query_trader_database(sql_query)

                # Realizamos la segunda llamada entregándole los datos verdaderos
                print("🧠 Alimentando al agente con datos reales de DuckDB para el veredicto final...")
                segundo_prompt = f"Here is the real data from the database for {ticker}:\n{datos_reales}\nBased on this real data, generate the final JSON recommendation."

                response = self.client.models.generate_content(
                    model=self.model_name,
                    contents=segundo_prompt,
                    config=types.GenerateContentConfig(system_instruction=TRADER_AGENT_PROMPT)
                )
                agent_verdict_json_str = response.text
            else:
                agent_verdict_json_str = primer_output

            # Guardar en caché el veredicto REAL final
            self._cache[cache_key] = agent_verdict_json_str
            self._save_cache()

            # --- PASO 3: Generación del Gráfico Integrado ---
            try:
                clean_agent_verdict = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_verdict)
                sql_query = f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker}' ORDER BY date DESC LIMIT 15"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]':
                    df_plot = pd.read_json(StringIO(data_json))
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)
                    df_plot['daily_range'] = df_plot['high'] - df_plot['low']

                    plt.figure(figsize=(12, 5))
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price')
                    plt.fill_between(df_plot['date'], df_plot['low'], df_plot['high'], color='skyblue', alpha=0.3, label='Volatility Range')
                    plt.title(f'{ticker} - Basic Analysis Price & Volatility', fontsize=14)
                    plt.grid(True, linestyle='--', alpha=0.5)
                    plt.xticks(rotation=30)
                    plt.tight_layout()
                    plt.show()
            except Exception as plot_e:
                print(f"⚠️ Nota sobre gráficos: {plot_e}")

            return agent_verdict_json_str
        except Exception as e:
            return f"❌ Error en el agente básico: {str(e)}"

    def analyze_advanced_indicators(self, ticker: str) -> str:
        print(f"🤖 Advanced AI agent initiating analysis for: {ticker}...")
        cache_key = f"advanced_analysis_{ticker}"
        if cache_key in self._cache:
            return self._cache[cache_key]

        # Para evitar alucinaciones, primero extraemos los datos de manera determinista en Python
        # y se los pasamos directamente consolidados al Agente Avanzado.
        try:
            print("📊 Extrayendo últimos 50 registros históricos de forma segura...")
            sql_query = f"SELECT date, open, high, low, close, volume FROM fact_historical_prices WHERE ticker = '{ticker}' ORDER BY date DESC LIMIT 50"
            data_json = query_trader_database(sql_query)

            if not data_json or data_json == '[]':
                return f"⚠️ No se encontraron datos para {ticker}"

            # Construimos el prompt enriquecido con los datos reales adjuntos
            prompt_con_datos = f"""
            Perform a detailed technical analysis for ticker '{ticker}'.
            Here is the factual historical data (last 50 records) from the database:
            {data_json}

            Evaluate the trend, calculate or interpret indicators like RSI and SMA, and return your final investment decision in JSON format.
            """

            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt_con_datos,
                config=types.GenerateContentConfig(system_instruction=ADVANCED_TRADER_AGENT_PROMPT)
            )
            agent_output_json_str = response.text

            # Almacenar en caché el reporte fundamentado en datos reales
            self._cache[cache_key] = agent_output_json_str
            self._save_cache()

            # --- Renderizado de Gráficos Técnicos Reales ---
            df_plot = pd.read_json(StringIO(data_json))
            df_plot['date'] = pd.to_datetime(df_plot['date'])
            df_plot = df_plot.sort_values('date').reset_index(drop=True)

            if len(df_plot) >= 20:
                df_plot['SMA_20'] = df_plot['close'].rolling(window=20).mean()
            if len(df_plot) >= 14:
                delta = df_plot['close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
                rs = gain / (loss + 1e-9)
                df_plot['rsi_14'] = 100 - (100 / (1 + rs))

            plt.figure(figsize=(14, 7))
            ax1 = plt.subplot(2, 1, 1)
            sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Price', ax=ax1)
            if 'SMA_20' in df_plot.columns:
                sns.lineplot(x='date', y='SMA_20', data=df_plot, color='red', label='SMA 20', ax=ax1)
            ax1.set_title(f'Advanced Technical Indicators: {ticker}', fontsize=14)
            ax1.grid(True, alpha=0.4)

            ax2 = plt.subplot(2, 1, 2, sharex=ax1)
            if 'rsi_14' in df_plot.columns:
                sns.lineplot(x='date', y='rsi_14', data=df_plot, color='green', label='RSI 14', ax=ax2)
            ax2.axhline(70, linestyle='--', color='red', alpha=0.6)
            ax2.axhline(30, linestyle='--', color='green', alpha=0.6)
            ax2.grid(True, alpha=0.4)
            plt.xticks(rotation=35)
            plt.tight_layout()
            plt.show()

            return agent_output_json_str
        except Exception as e:
            return f"❌ Error en agente avanzado: {str(e)}"

    def manage_portfolio_decision(self, ticker: str) -> str:
        print(f"\n🧠 Portfolio Manager Agent synthesizing decisions for: {ticker}...")
        trader_report = self.analyze_asset(ticker)
        advanced_trader_report = self.analyze_advanced_indicators(ticker)

        trader_recommendation, trader_justification = "HOLD", "No data"
        advanced_recommendation, advanced_justification = "HOLD", "No data"

        try:
            c_trader = trader_report.replace('```json', '').replace('```', '').strip()
            o_trader = json.loads(c_trader)
            trader_recommendation = o_trader.get('recommendation', o_trader.get('action', 'HOLD'))
            trader_justification = o_trader.get('technical_justification', 'Processed')
        except: pass

        try:
            c_adv = advanced_trader_report.replace('```json', '').replace('```', '').strip()
            o_adv = json.loads(c_adv)
            advanced_recommendation = o_adv.get('recommendation', 'HOLD')
            advanced_justification = o_adv.get('justification', o_adv.get('technical_analysis_report', 'Processed'))
        except: pass

        portfolio_manager_prompt_user = f"""
        Analyze the following reports for ticker '{ticker}':

        BASIC AGENT: Recommendation [{trader_recommendation}], Justification [{trader_justification}]
        ADVANCED AGENT: Recommendation [{advanced_recommendation}], Justification [{advanced_justification}]

        Provide the final consolidated investment decision.
        """
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=portfolio_manager_prompt_user,
                config=types.GenerateContentConfig(system_instruction=PORTFOLIO_MANAGER_AGENT_PROMPT)
            )
            decision = response.text.replace('```json', '').replace('```', '').strip()
            print("\n================ PORTFOLIO MANAGER AGENT'S DECISION ==================")
            print(decision)
            print("======================================================================")
            return decision
        except Exception as e:
            return f"❌ Error en Portfolio Manager: {str(e)}"

# Ejecución del pipeline corregido
if execute_agents:
    agente = AgentTraderPlatform()
    ticker_a_probar = ["TSLA"]
    for t in ticker_a_probar:
        print(f"\n--- INICIANDO ALINEACIÓN PARA: {t} ---")
        agente.manage_portfolio_decision(t)

Cache loaded from /content/drive/MyDrive/Trading_Agent_POCv1/agent_cache.json

--- INICIANDO ALINEACIÓN PARA: TSLA ---

🧠 Portfolio Manager Agent synthesizing decisions for: TSLA...
🤖 Basic AI agent initiating technical analysis for: TSLA...
Retrieving basic analysis for TSLA from cache.
🤖 Advanced AI agent initiating analysis for: TSLA...


**4.8 Agent Implementation and Decision Orchestration with Caching (Result Persistence) and Contingency for Gemini model service API, 2 layers: Copilot + Lima 3**

In [30]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import os

# Libraries required for contingency fallbacks
# Make sure you have installed: pip install google-genai openai huggingface_hub
from google import genai
from google.genai import types
import openai
from huggingface_hub import InferenceClient

class AgentTraderPlatform:
    def __init__(self):
        # 1. Initialization of the Main Client: Google GenAI
        self.client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        self.model_name = "gemini-2.5-flash"

        # 2. Initialization of Contingency Client 1: Copilot Ecosystem (OpenAI GPT-4o)
        # You can use a traditional OpenAI API Key or the GitHub Models token (Copilot Developer)
        self.copilot_client = openai.OpenAI(
            base_url="https://models.inference.ai.azure.com", # URL for GitHub Models / Copilot Ecosystem
            api_key=userdata.get('GITHUB_TOKEN')
        )
        self.copilot_model = "gpt-4o"

        #3. Contingency Client Initialization 2: Hugging Face Serverless Framework
        # The Llama-3 model with 70 Billion parameters is consumed without downloading weights locally
        self.hf_client = InferenceClient(
            model="meta-llama/Meta-Llama-3-70B-Instruct",
            token=userdata.get('HF_TOKEN')
        )

        self._cache = {}
        self._cache_file_path = os.path.join(BASE_PATH, 'agent_cache.json')
        self._load_cache()

    def _load_cache(self):
        if os.path.exists(self._cache_file_path):
            try:
                with open(self._cache_file_path, 'r', encoding='utf-8') as f:
                    self._cache = json.load(f)
                print(f"Cache loaded from {self._cache_file_path}")
            except json.JSONDecodeError as e:
                print(f"Error decoding cache file: {e}. Starting with empty cache.")
                self._cache = {}
            except Exception as e:
                print(f"Error loading cache file: {e}. Starting with empty cache.")
                self._cache = {}
        else:
            print("No cache file found. Starting with empty cache.")

    def _save_cache(self):
        try:
            os.makedirs(os.path.dirname(self._cache_file_path), exist_ok=True)
            with open(self._cache_file_path, 'w', encoding='utf-8') as f:
                json.dump(self._cache, f, indent=4)
            print(f"Cache saved to {self._cache_file_path}")
        except Exception as e:
            print(f"Error saving cache file: {e}")

    def _generate_content_with_fallback(self, prompt: str, system_instruction: str, tools=None) -> str:
        """
        Central Inference Orchestrator with Resilience Matrix and Automatic Fallback.
        Execution flow: Gemini-2.5-Flash -> Copilot (GPT-4o) -> Hugging Face (Llama-3-70B)
        """
        # --- LAYER 1: Main Attempt with Gemini ---
        try:
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    tools=tools
                )
            )
            return response.text
        except Exception as gemini_error:
            print(f"⚠️ Quota exceeded or error in Gemini. Activating contingency plan... Details: {gemini_error}")

            # --- LAYER 2: First Fallback - Copilot Ecosystem (GPT-4o) ---
            try:
                print(f"🔀 Connecting with Copilot Engine ({self.copilot_model})...")
                messages = [
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": prompt}
                ]
                # Note: DuckDB's native tools are omitted in plain text fallbacks to ensure compatibility
                copilot_response = self.copilot_client.chat.completions.create(
                    model=self.copilot_model,
                    messages=messages,
                    temperature=0.2
                )
                return copilot_response.choices[0].message.content
            except Exception as copilot_error:
                print(f"⚠️ Error or quota limit reached in Copilot Engine. Escalating to Hugging Face... Details: {copilot_error}")

                # --- LAYER 3: Second Fallback - Hugging Face Agents/Inference API ---
                try:
                    print("🔀 Connecting with Hugging Face Serverless Infrastructure (Llama-3-70B)...")
                    # Prompt formatted with embedded system instructions
                    formatted_prompt = f"<|system|>\n{system_instruction}\n<|user|>\n{prompt}\n<|assistant|>\n"

                    hf_response = self.hf_client.text_generation(
                        prompt=formatted_prompt,
                        max_new_tokens=1024,
                        temperature=0.2
                    )
                    return hf_response
                except Exception as hf_error:
                    # If the entire contingency ecosystem fails catastrophically
                    raise RuntimeError(f"❌ Critical multi-cloud failure. All providers.  (Gemini, Copilot, HF): {hf_error}")

    def analyze_asset(self, ticker: str) -> str:
        print(f"🤖 Basic AI agent initiating technical analysis for: {ticker}...")
        cache_key = f"basic_analysis_{ticker}"

        if cache_key in self._cache:
            print(f"Retrieving basic analysis for {ticker} from cache.")
            return self._cache[cache_key]

        prompt_usuario = f"""
        Perform an analytical analysis of the asset with ticker '{ticker}'.
        Consult its last 15 price records in the 'fact_historical_prices' table
        to evaluate its recent behavior, including price, volume, and volatility.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """

        try:
            # We consume the orchestrator with integrated fallback policies
            agent_verdict_json_str = self._generate_content_with_fallback(
                prompt=prompt_usuario,
                system_instruction=TRADER_AGENT_PROMPT,
                tools=[query_trader_database]
            )

            self._cache[cache_key] = agent_verdict_json_str
            self._save_cache()

            # --- Integrated Graphics Generation ---
            try:
                clean_agent_verdict = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_verdict)

                ticker_to_plot = agent_output.get('ticker', ticker)
                sql_query = f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT 15"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]':
                    df_plot = pd.read_json(StringIO(data_json))
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)
                    df_plot['daily_range'] = df_plot['high'] - df_plot['low']

                    plt.figure(figsize=(14, 7))
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price')
                    plt.fill_between(df_plot['date'], df_plot['low'], df_plot['high'], color='skyblue', alpha=0.3, label='Daily Price Range (Volatility)')
                    plt.title(f'{ticker_to_plot} Closing Prices and Daily Volatility (Last 15 Days)', fontsize=16)
                    plt.xlabel('Date', fontsize=12)
                    plt.ylabel('Price', fontsize=12)
                    plt.xticks(rotation=45)
                    plt.grid(True, linestyle='--', alpha=0.7)
                    plt.legend()
                    plt.tight_layout()
                    plt.show()
                    print(f"📈 Chart generated for {ticker_to_plot}.")
                else:
                    print(f"⚠️ No historical data available to plot for {ticker_to_plot}.")
            except json.JSONDecodeError:
                print(f"⚠️ Could not parse agent's JSON output for plotting.")
            except Exception as plot_e:
                print(f"❌ Error during chart generation: {plot_e}")

            return agent_verdict_json_str
        except Exception as e:
            return f"❌ Error in agent execution: {str(e)}"

    def analyze_advanced_indicators(self, ticker: str) -> str:
        print(f"🤖 Advanced AI agent initiating advanced indicator analysis for: {ticker}...")
        cache_key = f"advanced_analysis_{ticker}"

        if cache_key in self._cache:
            print(f"Retrieving advanced analysis for {ticker} from cache.")
            return self._cache[cache_key]

        prompt_usuario_advanced = f"""
        Perform a detailed analysis of the asset with ticker '{ticker}'.
        Consult its last 50 price records in the 'fact_historical_prices' table
        to evaluate its performance using technical indicators such as the RSI, SMA, and volume.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """
        try:
            # We consume the orchestrator with integrated fallback policies
            agent_output_json_str = self._generate_content_with_fallback(
                prompt=prompt_usuario_advanced,
                system_instruction=ADVANCED_TRADER_AGENT_PROMPT,
                tools=[query_trader_database]
            )

            self._cache[cache_key] = agent_output_json_str
            self._save_cache()

            try:
                clean_agent_output = agent_output_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_output)

                ticker_to_plot = agent_output.get('ticker', ticker)
                analysis_period = agent_output.get('analysis_period_days', 50)

                sql_query = f"SELECT date, open, high, low, close, volume FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT {analysis_period}"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]':
                    df_plot = pd.read_json(StringIO(data_json))
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)

                    if not df_plot.empty and len(df_plot) >= 20:
                        df_plot['SMA_20'] = df_plot['close'].rolling(window=20).mean()
                    if not df_plot.empty and len(df_plot) >= 14:
                        delta = df_plot['close'].diff()
                        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
                        rs = gain / loss
                        df_plot['rsi_14'] = 100 - (100 / (1 + rs))

                    plt.figure(figsize=(16, 9))
                    ax1 = plt.subplot(2, 1, 1)
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price', ax=ax1)
                    if 'SMA_20' in df_plot.columns:
                        sns.lineplot(x='date', y='SMA_20', data=df_plot, color='red', label='SMA 20', linestyle='--', ax=ax1)
                    ax1.set_title(f'Prices and Technical Indicators of {ticker_to_plot} (Latest {analysis_period} Days)', fontsize=16)
                    ax1.set_ylabel('Price', fontsize=12)
                    ax1.legend()
                    ax1.grid(True, linestyle='--', alpha=0.7)
                    plt.setp(ax1.get_xticklabels(), visible=False)

                    ax2 = plt.subplot(2, 1, 2, sharex=ax1)
                    if 'rsi_14' in df_plot.columns:
                        sns.lineplot(x='date', y='rsi_14', data=df_plot, color='green', label='RSI 14', ax=ax2)
                    ax2.axhline(70, linestyle='--', alpha=0.5, color='red')
                    ax2.axhline(30, linestyle='--', alpha=0.5, color='green')
                    ax2.set_title('RSI (Relative Strength Index)', fontsize=14)
                    ax2.set_ylabel('Indicator Value', fontsize=12)
                    ax2.set_xlabel('Date', fontsize=12)
                    ax2.grid(True, linestyle='--', alpha=0.7)
                    ax2.legend(['Overbought (70)', 'Oversold (30)'])
                    plt.xticks(rotation=45)
                    plt.tight_layout()
                    plt.show()
                    print(f"📈 Advanced Indicators Chart generated for {ticker_to_plot}.")
            except json.JSONDecodeError:
                print(f"⚠️ The JSON output from the agent could not be parsed for graphing.")

            return agent_output_json_str
        except Exception as e:
            return f"❌ Error in the execution of the advanced analysis agent: {str(e)}"

    def manage_portfolio_decision(self, ticker: str) -> str:
        print(f"🧠 Portfolio Manager Agent synthesizing decisions for: {ticker}...")

        trader_report = self.analyze_asset(ticker)
        advanced_trader_report = self.analyze_advanced_indicators(ticker)

        trader_recommendation, trader_justification = "N/A", "N/A"
        advanced_trader_recommendation, advanced_trader_justification = "N/A", "N/A"

        try:
            clean_trader_report = trader_report.replace('```json', '').replace('```', '').strip()
            trader_output = json.loads(clean_trader_report)
            trader_recommendation = trader_output.get('recommendation', 'N/A')
            trader_justification = trader_output.get('technical_justification', 'N/A')
        except Exception as e:
            print(f"❌ Error processing TRADER_AGENT_PROMPT output: {str(e)}")

        try:
            clean_advanced_trader_report = advanced_trader_report.replace('```json', '').replace('```', '').strip()
            advanced_trader_output = json.loads(clean_advanced_trader_report)
            advanced_trader_recommendation = advanced_trader_output.get('recommendation', 'N/A')
            advanced_trader_justification = advanced_trader_output.get('justification', 'N/A')
        except Exception as e:
            print(f"❌ Error processing ADVANCED_TRADER_AGENT_PROMPT output: {str(e)}")

        portfolio_manager_prompt_user = f"""
        Analyze the following reports for ticker '{ticker}':

        TRADER_AGENT_PROMPT Report:
        Recommendation: {trader_recommendation}
        Justification: {trader_justification}

        ADVANCED_TRADER_AGENT_PROMPT Report:
        Recommendation: {advanced_trader_recommendation}
        Justification: {advanced_trader_justification}

        Based on these inputs, provide a final consolidated investment decision.
        """

        try:
            # We consume the orchestrator with integrated fallback policies
            portfolio_decision_json_str = self._generate_content_with_fallback(
                prompt=portfolio_manager_prompt_user,
                system_instruction=PORTFOLIO_MANAGER_AGENT_PROMPT
            )

            clean_portfolio_decision = portfolio_decision_json_str.replace('```json', '').replace('```', '').strip()

            print("\n================ PORTFOLIO MANAGER AGENT'S DECISION ==================")
            print(clean_portfolio_decision)
            print("======================================================================")

            return clean_portfolio_decision
        except Exception as e:
            return f"❌ Error in Portfolio Manager Agent execution: {str(e)}"


# =====================================================================
# SPRINT 4 EXECUTION AND TESTING (CORREGIDO)
# =====================================================================
agente = AgentTraderPlatform()

# SOLUCIONADO EL BUG DEL BUCLE: Declarado explícitamente como una lista para evitar iteración por caracteres
tickers_a_probar = ["TSLA"]

for ticker_actual in tickers_a_probar:
    print(f"\n\n--- STARTING ANALYSIS FOR THE ASSET: {ticker_actual} ---")

    agent_report = agente.analyze_asset(ticker_actual)
    print("\n================ AI AGENT'S REPORT ==================")
    print(agent_report)
    print("======================================================================")

    print("\n================ TESTING NEW ADVANCED ANALYTICS AGENT ==================")
    advanced_agent_report = agente.analyze_advanced_indicators(ticker_actual)
    print("\n================ ADVANCED ANALYSIS AGENT'S REPORT ==================")
    print(advanced_agent_report)
    print("======================================================================")

    portfolio_decision = agente.manage_portfolio_decision(ticker_actual)

Cache loaded from /content/drive/MyDrive/Trading_Agent_POCv1/agent_cache.json


--- STARTING ANALYSIS FOR THE ASSET: TSLA ---
🤖 Basic AI agent initiating technical analysis for: TSLA...
Retrieving basic analysis for TSLA from cache.

================ AI AGENT'S REPORT ==================
```json
{
  "ticker": "TSLA",
  "trend_analysis": "Over the analyzed period, TSLA's price has shown a downward trend from a high of $445.27 to a recent close of $423.74. There is significant volatility, with daily price ranges often exceeding $10. Volume has been relatively high, indicating strong market activity, but the price trend suggests selling pressure.",
  "volatility_analysis": "The volatility is notable, with an average daily range of $13.57 (calculated as the average of daily high-low differences). This indicates significant price fluctuations, which could signal uncertainty or active trading. The downward trend in closing prices suggests that the volatility is skewed towards the downside.",


**4.8.a Agent Implementation and Decision Orchestration with Caching (Result Persistence), Contingency for Gemini model service API, 2 layers: Copilot + Lima 3 and ReAct Flow, RAG control and Clean Consolidation**

In [31]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from io import StringIO
import os
import sys

# Libraries required for contingency fallbacks
from google import genai
from google.genai import types
import openai
from huggingface_hub import InferenceClient

class AgentTraderPlatform:
    def __init__(self):
        print("🔍 [DEBUG 1] Starting the Agent constructor __init__...")

        # Validate critical environment variables before connecting
        if 'BASE_PATH' not in globals():
            print("❌ [CRITICAL ERROR] 'BASE_PATH' It is not defined in the global environment.")
        if 'TRADER_AGENT_PROMPT' not in globals():
            print("❌ [CRITICAL ERROR] 'TRADER_AGENT_PROMPT' It is not defined.")

        # 1. Initialization of the Main Client: Google GenAI
        print("🔍 [DEBUG 2] Connecting with Google GenAI Client...")
        self.client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
        self.model_name = "gemini-2.5-flash"

        # 2. Initialization of Contingency Client 1: Copilot Ecosystem (OpenAI GPT-4o)
        print("🔍 [DEBUG 3] Connecting with Copilot Engine Ecosystem...")
        self.copilot_client = openai.OpenAI(
            base_url="https://models.inference.ai.azure.com",
            api_key=userdata.get('GITHUB_TOKEN')
        )
        self.copilot_model = "gpt-4o"

        # 3. Contingency Client Initialization 2: Hugging Face Serverless Framework
        print("🔍 [DEBUG 4] Connecting with Hugging Face Client...")
        self.hf_client = InferenceClient(
            model="meta-llama/Meta-Llama-3-70B-Instruct",
            token=userdata.get('HF_TOKEN')
        )

        self._cache = {}
        self._cache_file_path = os.path.join(BASE_PATH, 'agent_cache.json')
        self._load_cache()

        # Invalidation of META cache
        #print("🔍 [DEBUG 5] Cleaning cached META records to ensure clean execution...")
        #for key in ["basic_analysis_META", "advanced_analysis_META"]:
        #    if key in self._cache:
        #        del self._cache[key]
        #self._save_cache()
        #print("🔍 [DEBUG 6] Constructor finalizado con éxito.")

    def _load_cache(self):
        if os.path.exists(self._cache_file_path):
            try:
                with open(self._cache_file_path, 'r', encoding='utf-8') as f:
                    self._cache = json.load(f)
                print(f"Cache loaded from {self._cache_file_path}")
            except Exception as e:
                print(f"Error loading cache file: {e}. Starting with empty cache.")
                self._cache = {}
        else:
            print("No cache file found. Starting with empty cache.")

    def _save_cache(self):
        try:
            os.makedirs(os.path.dirname(self._cache_file_path), exist_ok=True)
            with open(self._cache_file_path, 'w', encoding='utf-8') as f:
                json.dump(self._cache, f, indent=4)
            print(f"Cache saved to {self._cache_file_path}")
        except Exception as e:
            print(f"Error saving cache file: {e}")

    def _generate_content_with_fallback(self, prompt: str, system_instruction: str, tools=None) -> str:
        """
        Central Inference Orchestrator with Resilience Matrix and Automatic Fallback.
        """
        # --- LAYER 1: Main Attempt with Gemini ---
        try:
            print(f"   📡 [LLM CALL] Sending request to LAYER 1: Gemini ({self.model_name})...")
            response = self.client.models.generate_content(
                model=self.model_name,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    tools=tools
                )
            )
            print("   ✅ [LLM CALL] LAYER 1 Responded successfully.")
            return response.text
        except Exception as gemini_error:
            print(f"   ⚠️ [FALLBACK] Gemini error: {gemini_error}. Switching to LAYER 2...")

            # --- LAYER 2: First Fallback - Copilot Ecosystem (GPT-4o) ---
            try:
                print(f"   📡 [LLM CALL] Sending request to LAYER 2: Copilot ({self.copilot_model})...")
                messages = [
                    {"role": "system", "content": system_instruction},
                    {"role": "user", "content": prompt}
                ]
                copilot_response = self.copilot_client.chat.completions.create(
                    model=self.copilot_model,
                    messages=messages,
                    temperature=0.2
                )
                print("   ✅ [LLM CALL] LAYER 2 Responded successfully.")
                return copilot_response.choices[0].message.content
            except Exception as copilot_error:
                print(f"   ⚠️ [FALLBACK] Copilot error: {copilot_error}. Switching to Layer 3...")

                # --- LAYER 3: Second Fallback - Hugging Face (Llama-3-70B) ---
                try:
                    print("   📡 [LLM CALL] Sending request to LAYER 3: Hugging Face (Llama-3)...")
                    formatted_prompt = f"<|system|>\n{system_instruction}\n<|user|>\n{prompt}\n<|assistant|>\n"
                    hf_response = self.hf_client.text_generation(
                        prompt=formatted_prompt,
                        max_new_tokens=1024,
                        temperature=0.2
                    )
                    print("   ✅ [LLM CALL] LAYER 3 Responded successfully.")
                    return hf_response
                except Exception as hf_error:
                    raise RuntimeError(f"❌ Critical multi-cloud failure. All providers failed: {hf_error}")

    def analyze_asset(self, ticker: str) -> str:
        print(f"\n🤖 [METHOD] Starting analyze_asset for: {ticker}...")
        cache_key = f"basic_analysis_{ticker}"

        if cache_key in self._cache:
            print(f"🎯 Retrieving {ticker} analysis from the cache.")
            return self._cache[cache_key]

        prompt_usuario = f"""
        Perform an analytical analysis of the asset with ticker '{ticker}'.
        Consult its last 15 price records in the 'fact_historical_prices' table
        to evaluate its recent behavior, including price, volume, and volatility.
        Generate your investment recommendation and create a line chart that visualizes this data.
        """

        try:
            print("🚀 [STEP 1] Executing first LLM call (ReAct Discovery)...")
            agent_verdict_json_str = self._generate_content_with_fallback(
                prompt=prompt_usuario,
                system_instruction=TRADER_AGENT_PROMPT,
                tools=[query_trader_database]
            )

            # COMPLETE REACT FLOW
            if "query_trader_database" in agent_verdict_json_str and "recommendation" not in agent_verdict_json_str:
                print("🔄 [REACT LOOP] The agent detected the native tool. Extracting SQL query...")

                try:
                    clean_json = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                    tool_call = json.loads(clean_json)
                    sql_query = tool_call.get("query")
                except Exception:
                    import re
                    match = re.search(r"SELECT.*?;", agent_verdict_json_str, re.IGNORECASE | re.DOTALL)
                    sql_query = match.group(0) if match else f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker}' ORDER BY date DESC LIMIT 15"

                print(f"📊 [DATABASE] Running a query on DuckDB: {sql_query}")
                datos_reales = query_trader_database(sql_query)
                print(f"📊 [DATABASE] Records retrieved (Character length): {len(datos_reales)})")

                print("🧠 [STEP 2] Reinjecting real data into the model for final report...")
                segundo_prompt = f"""
                Here is the real factual data from the database for {ticker} to complete your analysis:
                {datos_reales}

                Based on this real data, generate your final investment recommendation following the exact JSON format specified.
                """
                agent_verdict_json_str = self._generate_content_with_fallback(
                    prompt=segundo_prompt,
                    system_instruction=TRADER_AGENT_PROMPT
                )

            self._cache[cache_key] = agent_verdict_json_str
            self._save_cache()

            # --- Generation of Graphics ---
            print("🎨 [GRAPHICS] Processing basic graphics rendering...")
            try:
                clean_agent_verdict = agent_verdict_json_str.replace('```json', '').replace('```', '').strip()
                agent_output = json.loads(clean_agent_verdict)
                ticker_to_plot = agent_output.get('ticker', ticker)

                sql_query = f"SELECT date, open, high, low, close FROM fact_historical_prices WHERE ticker = '{ticker_to_plot}' ORDER BY date DESC LIMIT 15"
                data_json = query_trader_database(sql_query)

                if data_json and data_json != '[]':
                    df_plot = pd.read_json(StringIO(data_json))
                    df_plot['date'] = pd.to_datetime(df_plot['date'])
                    df_plot = df_plot.sort_values('date').reset_index(drop=True)
                    df_plot['daily_range'] = df_plot['high'] - df_plot['low']

                    plt.figure(figsize=(14, 7))
                    sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price')
                    plt.fill_between(df_plot['date'], df_plot['low'], df_plot['high'], color='skyblue', alpha=0.3, label='Daily Price Range (Volatility)')
                    plt.title(f'{ticker_to_plot} Closing Prices and Daily Volatility (Last 15 Days)', fontsize=16)
                    plt.xticks(rotation=45)
                    plt.grid(True, linestyle='--', alpha=0.7)
                    plt.legend()
                    plt.tight_layout()
                    plt.show()
                    print(f"📈 Chart generated successfully for {ticker_to_plot}.")
                else:
                    print(f"⚠️ There is no historical data to graph basic {ticker_to_plot}.")
            except Exception as plot_e:
                print(f"❌ Alert in basic graphics rendering: {plot_e}")

            return agent_verdict_json_str
        except Exception as e:
            return f"❌ Error in the execution of the basic agent: {str(e)}"

    def analyze_advanced_indicators(self, ticker: str) -> str:
        print(f"\n🤖 [METHOD] Starting analyze_advanced_indicators for: {ticker}...")
        cache_key = f"advanced_analysis_{ticker}"

        if cache_key in self._cache:
            print(f"🎯 Retrieving advanced analysis of {ticker} from the cache.")
            return self._cache[cache_key]

        try:
            print("📊 [DATABASE] Proactively extracting the last 50 records to avoid hallucinations...")
            sql_query = f"SELECT date, open, high, low, close, volume FROM fact_historical_prices WHERE ticker = '{ticker}' ORDER BY date DESC LIMIT 50"
            data_json = query_trader_database(sql_query)

            if not data_json or data_json == '[]':
                print(f"⚠️ Empty or recordless database for {ticker}.")
                return f'{{"ticker": "{ticker}", "recommendation": "HOLD", "justification": "No data available."}}'

            prompt_usuario_advanced = f"""
            Perform a detailed analysis of the asset with ticker '{ticker}'.
            Here is the factual historical data (last 50 records) pulled directly from our analytical datalake:
            {data_json}

            Evaluate its performance using technical indicators such as the RSI, SMA, and volume based STRICTLY on the real records provided above.
            Generate your investment recommendation and structure the response in JSON format.
            """

            print("🚀 [STEP 1] Sending deterministic data to the LLM matrix...")
            agent_output_json_str = self._generate_content_with_fallback(
                prompt=prompt_usuario_advanced,
                system_instruction=ADVANCED_TRADER_AGENT_PROMPT
            )

            self._cache[cache_key] = agent_output_json_str
            self._save_cache()

            # --- Advanced Graphics ---
            print("🎨 [GRAPHING MAKER] Processing rendering of advanced technical indicators...")
            try:
                df_plot = pd.read_json(StringIO(data_json))
                df_plot['date'] = pd.to_datetime(df_plot['date'])
                df_plot = df_plot.sort_values('date').reset_index(drop=True)

                if not df_plot.empty and len(df_plot) >= 20:
                    df_plot['SMA_20'] = df_plot['close'].rolling(window=20).mean()
                if not df_plot.empty and len(df_plot) >= 14:
                    delta = df_plot['close'].diff()
                    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
                    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
                    rs = gain / (loss + 1e-9)
                    df_plot['rsi_14'] = 100 - (100 / (1 + rs))

                plt.figure(figsize=(16, 9))
                ax1 = plt.subplot(2, 1, 1)
                sns.lineplot(x='date', y='close', data=df_plot, marker='o', color='blue', label='Closing Price', ax=ax1)
                if 'SMA_20' in df_plot.columns:
                    sns.lineplot(x='date', y='SMA_20', data=df_plot, color='red', label='SMA 20', linestyle='--', ax=ax1)
                ax1.set_title(f'Prices and Technical Indicators of {ticker} (Latest 50 Days)', fontsize=16)
                ax1.grid(True, linestyle='--', alpha=0.7)
                plt.setp(ax1.get_xticklabels(), visible=False)

                ax2 = plt.subplot(2, 1, 2, sharex=ax1)
                if 'rsi_14' in df_plot.columns:
                    sns.lineplot(x='date', y='rsi_14', data=df_plot, color='green', label='RSI 14', ax=ax2)
                ax2.axhline(70, linestyle='--', alpha=0.5, color='red')
                ax2.axhline(30, linestyle='--', alpha=0.5, color='green')
                ax2.grid(True, linestyle='--', alpha=0.7)
                plt.xticks(rotation=45)
                plt.tight_layout()
                plt.show()
                print(f"📈 Advanced Indicators Chart generated for {ticker}.")
            except Exception as graph_err:
                print(f"⚠️ Alert by graphing advanced indicators: {graph_err}")

            return agent_output_json_str
        except Exception as e:
            return f"❌ Error in the execution of the advanced agent: {str(e)}"

    def manage_portfolio_decision(self, ticker: str) -> str:
        print(f"\n🧠 [METHOD] Portfolio Manager Agent synthesizing decisions for: {ticker}...")

        trader_report = self.analyze_asset(ticker)
        advanced_trader_report = self.analyze_advanced_indicators(ticker)

        trader_recommendation, trader_justification = "HOLD", "No data"
        advanced_trader_recommendation, advanced_trader_justification = "HOLD", "No data"

        try:
            clean_trader_report = trader_report.replace('```json', '').replace('```', '').strip()
            trader_output = json.loads(clean_trader_report)
            trader_recommendation = trader_output.get('recommendation', trader_output.get('action', 'HOLD'))
            trader_justification = trader_output.get('technical_justification', trader_output.get('justification', 'Processed.'))
        except Exception as e:
            print(f"⚠️ Minor adaptation basic mapping: {str(e)}")

        try:
            clean_advanced_trader_report = advanced_trader_report.replace('```json', '').replace('```', '').strip()
            advanced_trader_output = json.loads(clean_advanced_trader_report)
            advanced_trader_recommendation = advanced_trader_output.get('recommendation', 'HOLD')
            advanced_trader_justification = advanced_trader_output.get('justification', advanced_trader_output.get('technical_analysis_report', 'Processed.'))
        except Exception as e:
            print(f"⚠️ Minor adaptation advanced mapping: {str(e)}")

        portfolio_manager_prompt_user = f"""
        Analyze the following reports for ticker '{ticker}':

        TRADER_AGENT_PROMPT Report:
        Recommendation: {trader_recommendation}
        Justification: {trader_justification}

        ADVANCED_TRADER_AGENT_PROMPT Report:
        Recommendation: {advanced_trader_recommendation}
        Justification: {advanced_trader_justification}

        Based on these inputs, provide a final consolidated investment decision.
        """

        try:
            print("🚀 [FINAL STEP] Requesting clean consolidation from the Portfolio Manager...")
            portfolio_decision_json_str = self._generate_content_with_fallback(
                prompt=portfolio_manager_prompt_user,
                system_instruction=PORTFOLIO_MANAGER_AGENT_PROMPT
            )

            clean_portfolio_decision = portfolio_decision_json_str.replace('```json', '').replace('```', '').strip()

            print("\n================ PORTFOLIO MANAGER AGENT'S DECISION ==================")
            print(clean_portfolio_decision)
            print("======================================================================")

            return clean_portfolio_decision
        except Exception as e:
            return f"❌ Portfolio Manager Agent error: {str(e)}"


# =====================================================================
# INTEGRATED EXECUTION BLOCK AND JSON DISPLAY
# =====================================================================
print("🏁 [MAIN] Initializing agent platform...")
agente = AgentTraderPlatform()

tickers_a_probar = ["NVDA"]
print(f"🏁 [MAIN] Tickers loaded for testing: {tickers_a_probar}")

for ticker_actual in tickers_a_probar:
    print(f"\n\n======================================================================")
    print(f"🚀 [MAIN] STARTING INTEGRATED ANALYSIS PIPELINE: {ticker_actual}")
    print(f"======================================================================")

    # 1. Implementation and Demonstration of the Basic Technical Agent (15 days + ReAct Loop)
    print("\n➡️ [STEP 1] Running Basic Technical Agent...")
    raw_basic_report = agente.analyze_asset(ticker_actual)

    print("\n================ 📄 AGENT BASIC TRADER JSON OUTPUT ================")
    # Visual cleanup in case it comes wrapped in markdown format ```json
    clean_basic = raw_basic_report.replace('```json', '').replace('```', '').strip()
    try:
        print(json.dumps(json.loads(clean_basic), indent=2, ensure_ascii=False))
    except Exception:
        print(raw_basic_report) # Fallback if it's not a valid JSON
    print("======================================================================")


    # 2. Execution and Demonstration of the Advanced Agent (50 days + Data Feeding)
    print("\n➡️ [STEP 2] Running Advanced Indicators Agent...")
    raw_advanced_report = agente.analyze_advanced_indicators(ticker_actual)

    print("\n================ 📄 ADVANCED ANALYTICS AGENT JSON OUTPUT ================")
    clean_advanced = raw_advanced_report.replace('```json', '').replace('```', '').strip()
    try:
        print(json.dumps(json.loads(clean_advanced), indent=2, ensure_ascii=False))
    except Exception:
        print(raw_advanced_report)
    print("======================================================================")


    # 3. Execution and Demonstration of the Final Consolidating Portfolio Manager
    print("\n➡️ [STEP 3] Requesting Strategic Consolidation from the Portfolio Manager...")
    #This method will consume the reports already saved in the cache extremely quickly.
    portfolio_decision = agente.manage_portfolio_decision(ticker_actual)

    print(f"\n✅ [MAIN] PIPELINE SUCCESSFULLY COMPLETED FOR: {ticker_actual}")

🏁 [MAIN] Initializing agent platform...
🔍 [DEBUG 1] Starting the Agent constructor __init__...
🔍 [DEBUG 2] Connecting with Google GenAI Client...
🔍 [DEBUG 3] Connecting with Copilot Engine Ecosystem...
🔍 [DEBUG 4] Connecting with Hugging Face Client...
Cache loaded from /content/drive/MyDrive/Trading_Agent_POCv1/agent_cache.json
🏁 [MAIN] Tickers loaded for testing: ['NVDA']


🚀 [MAIN] STARTING INTEGRATED ANALYSIS PIPELINE: NVDA

➡️ [STEP 1] Running Basic Technical Agent...

🤖 [METHOD] Starting analyze_asset for: NVDA...
🎯 Retrieving NVDA analysis from the cache.

================ 📄 AGENT BASIC TRADER JSON OUTPUT ================
{
  "ticker": "NVDA",
  "trend_analysis": "NVDA has shown a mixed trend over the analyzed period. The stock experienced a peak closing price of $235.74, followed by a decline to $222.82. There is evidence of short-term volatility, with significant fluctuations in both price and volume. The volume has been relatively high, indicating strong market activity, but 

# **5. Deployment to Production via WebApp**

We will use **FastAPI** for the *backend*, **Streamlit** for the *frontend*, and a *secure tunne*l with **Localtunnel**.

**5.1 Installation of Deployment Dependencies**

In [32]:
# Install the frameworks for the App and API
!pip install fastapi uvicorn streamlit pydantic requests

# Install localtunnel globally using Node.js to expose Colab ports
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 45.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙
added 22 packages in 3s
⠙
⠙3 packages are looking for funding
⠙  run `npm fund` for details
⠙

**5.2 Creating the Agent Backend with FastAPI (main.py)**

This **API** will expose **two main endpoint**s: one to query the generated **trading signals** (integrating the caching or database configured in BASE_PATH) and another to simulate **interactive chat** with the **multi-agent system.**

In [33]:
%%writefile main.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import os
import json

app = FastAPI(
    title="AlphaAgent SaaS API",
    description="High-availability multi-agent backend for signal consumption (Basic, Advanced and Portfolio Manager)",
    version="1.5"
)

class ChatRequest(BaseModel):
    message: str

# System persistent base path
BASE_PATH = '/content/drive/MyDrive/Trading_Agent_POCv1'

@app.get("/api/signals/{ticker}")
def get_trading_signal(ticker: str):
    """
    High-fidelity endpoint that reads the 'agent_cache.json' file generated in Sprint 4
    and extracts analytical metrics from all 3 layers of the agent system
    """
    ticker_upper = ticker.upper()
    cache_path = os.path.join(BASE_PATH, "agent_cache.json")

    if os.path.exists(cache_path):
        try:
            with open(cache_path, 'r', encoding='utf-8') as f:
                cache_data = json.load(f)

            # Key construction according to the exact nomenclature of Cell 55
            basic_key = f"basic_analysis_{ticker_upper}"
            advanced_key = f"advanced_analysis_{ticker_upper}"

            basic_raw = cache_data.get(basic_key, None)
            advanced_raw = cache_data.get(advanced_key, None)

            if basic_raw or advanced_raw:
                # 1. Secure parsing of Basic AI Agent data
                basic_parsed = {}
                if basic_raw:
                    try:
                        clean_str = basic_raw.replace('```json', '').replace('```', '').strip()
                        basic_parsed = json.loads(clean_str)
                    except:
                        basic_parsed = {"recommendation": "ERROR", "technical_justification": basic_raw}

                # 2. Secure parsing of Advanced AI Agent data
                advanced_parsed = {}
                if advanced_raw:
                    try:
                        clean_str = advanced_raw.replace('```json', '').replace('```', '').strip()
                        advanced_parsed = json.loads(clean_str)
                    except:
                        advanced_parsed = {"recommendation": "ERROR", "justification": advanced_raw}

                # Extraction of baseline recommendations
                rec_basic = basic_parsed.get("recommendation", "HOLD") if isinstance(basic_parsed, dict) else "HOLD"
                rec_adv = advanced_parsed.get("recommendation", "HOLD") if isinstance(advanced_parsed, dict) else "HOLD"
                just_basic = basic_parsed.get("technical_justification", "Trend analysis completed.") if isinstance(basic_parsed, dict) else str(basic_raw)
                just_adv = advanced_parsed.get("justification", "Calculation and interpretation of RSI, SMA and Volatility completed.") if isinstance(advanced_parsed, dict) else str(advanced_raw)

                # 3. Synthetic logic for the Portfolio Manager Agent based on consensus/discrepancy
                if rec_basic == rec_adv:
                    pm_rec = rec_basic
                    pm_just = f"Absolute consensus of the agent committee. Both sub-agents agree on a {pm_rec} strategy for protection and optimization."
                else:
                    pm_rec = rec_adv  # The advanced agent brings greater analytical depth
                    pm_just = f"A discrepancy is detected between the systems. The Basic Agent suggests {rec_basic}, but the Portfolio Manager prioritizes the {rec_adv} action dictated by the Advanced Agent due to the strength of the technical analysis indicators indexed in DuckDB."

                return {
                    "ticker": ticker_upper,
                    "has_data": True,
                    "basic_agent": {"recommendation": rec_basic, "justification": just_basic},
                    "advanced_agent": {"recommendation": rec_adv, "justification": just_adv},
                    "portfolio_manager": {"recommendation": pm_rec, "justification": pm_just}
                }
        except Exception as e:
            raise HTTPException(status_code=500, detail=f"Internal error reading persistent cache: {str(e)}")

    # Structured response Fallback if the asset has not executed in the main loop
    return {
        "ticker": ticker_upper,
        "has_data": False,
        "basic_agent": {"recommendation": "HOLD", "justification": "Active in market processing queue."},
        "advanced_agent": {"recommendation": "HOLD", "justification": "Pending historical volume indexing in DuckDB."},
        "portfolio_manager": {"recommendation": "HOLD", "justification": f"The asset {ticker_upper} has no records in 'agent_cache.json'. Run your agents in the control block to persist the rulings."}
    }

@app.post("/api/chat")
def agent_chat(request: ChatRequest):
    """
    Conversational endpoint tailored to simulate cross-responses and interpretations
from all three logical layers of your agent infrastructure.
    """
    user_msg = request.message.lower()

    if "apple" in user_msg or "aapl" in user_msg:
        return {"response": "🤖 **Basic AI Agent:** AAPL is showing bullish consolidation above its 20-day SMA.\n\n📊 **Advanced AI Agent:** Overbought alerts detected (RSI near 68).\n\n🧠 **Portfolio Manager Agent:** I recommend maintaining current positions and limiting further purchases until a healthy correction."}
    elif "risk" in user_msg or "volatility" in user_msg:
        return {"response": "🛡️ **Portfolio Manager:** Currently, the thresholds remain under structural control."}
    elif "lista" in user_msg or "tickers" in user_msg:
        return {"response": "📁 The ecosystem currently queries and writes centrally to the Drive Data Lake through the unified `agent_cache.json` file."}
    else:
        return {"response": f"I have received your request: '{request.message}'. The 3 agents (Basic, Advanced and Portfolio Manager) are ready to process this request in the next database synchronization cycle."}

Writing main.py


**5.3 Develop the Interactive Interface with Streamlit (app.py)**

In [34]:
%%writefile app.py
import streamlit as st
import requests
import pandas as pd

st.set_page_config(page_title="AlphaAgent SaaS - Multi-Agent Dashboard", layout="wide", page_icon="📈")

st.title("📈 AlphaAgent — Multi-Agent Trading Platform")
st.caption("Visual Modules for Asset Monitoring, Portfolio Monitoring and Interactive Chat (LLM)")
st.markdown("---")

API_URL = "http://localhost:8000"

# --- SIDEBAR: Dynamic Selection ---
st.sidebar.header("⚙️ Control Panel")

# 1. Definition of the set of 30 assets
AVAILABLE_ASSETS = [
    "AAPL", "MSFT", "AMZN", "GOOGL", "META", "TSLA", "NVDA", "NFLX", "AMD", "INTC",
    "ECOPETROL", "PFBCOLOM", "BVC", "GRUPOSURA", "NUTRESA", "JPM", "BAC", "WMT",
    "PG", "XOM", "CVX", "JNJ", "PFE", "V", "MA", "DIS", "NKE", "HD", "COST", "PEP"
]

# 2. Quick option to select the entire universe of assets
select_all = st.sidebar.checkbox("🌍 Select all 30 assets")

# 3. Ticker list assignment logic
if select_all:
    # If you check the box, all assets will be automatically selected
    tickers_list = AVAILABLE_ASSETS
    st.sidebar.info(f"Selected: All {len(AVAILABLE_ASSETS)} assets.")
else:
    # Otherwise, allow a custom selection using multiselect
    tickers_list = st.sidebar.multiselect(
        "Select the tickers to query:",
        options=AVAILABLE_ASSETS,
        default=["AAPL", "MSFT", "AMZN", "GOOGL"]  # Default initial values
    )

# Keep the chat history initialization under the selection
if "chat_history" not in st.session_state:
    st.session_state.chat_history = []

# --- VISUAL MODULES (Updated Sprint 5 Tabs) ---
tab_monitor, tab_risk, tab_chat = st.tabs([
    "🖥️ Multi-Agent Asset Monitor",
    "🛡️ Portfolio Monitoring Panel",
    "💬 Interactive Chat (LLM)"
])

# 1. MULTI-AGENT ASSET MONITOR (Layered breakdown of analysis)
with tab_monitor:
    st.subheader("🔍 Recommendation from each agent")
    st.markdown("Individual recommendations from the **Basic AI Agent**, **Advanced AI Agent**, and final decision from the **Portfolio Manager Agent**:")

    for ticker in tickers_list:
        with st.expander(f"📊 Multi-Agent Technician Report: {ticker}", expanded=True):
            try:
                response = requests.get(f"{API_URL}/api/signals/{ticker}")
                if response.status_code == 200:
                    data = response.json()

                    if data.get("has_data", False):
                        col1, col2, col3 = st.columns(3)

                        with col1:
                            st.markdown("### 🤖 Basic AI Agent")
                            rec = data["basic_agent"]["recommendation"]
                            st.metric(label="Technical Recommendation", value=rec)
                            st.caption(data["basic_agent"]["justification"])

                        with col2:
                            st.markdown("### 📊 Advanced AI Agent")
                            rec_adv = data["advanced_agent"]["recommendation"]
                            st.metric(label="Advanced Indicators", value=rec_adv)
                            st.caption(data["advanced_agent"]["justification"])

                        with col3:
                            st.markdown("### 🧠 Portfolio Manager")
                            rec_pm = data["portfolio_manager"]["recommendation"]

                            # Conditional marker according to the final recommendation
                            if "BUY" in rec_pm.upper():
                                st.success(f"🟢 recommendation: {rec_pm}")
                            elif "SELL" in rec_pm.upper():
                                st.error(f"🔴 recommendation: {rec_pm}")
                            else:
                                st.warning(f"🟡 recommendation: {rec_pm}")

                            st.info(data["portfolio_manager"]["justification"])
                    else:
                        st.warning(f"⚠️ {data['portfolio_manager']['justification']}")

            except Exception as e:
                st.error(f"❌ Critical infrastructure error: Unable to connect to the API backend. Details: {str(e)}")

# 2. PORTFOLIO MONITORING PANEL
with tab_risk:
    st.subheader("🛡️ Trading rules")

    col_r1, col_r2, col_r3 = st.columns(3)
    with col_r1:
        st.metric(label="Suggested Capital Exposure (Max)", value="18.600.000", delta="15% this week")
    with col_r2:
        st.metric(label="Status Control Rules (DuckDB)", value="LOW RISK", delta_color="inverse")
    with col_r3:
        st.metric(label="Recorded Maximum Drawdown", value="1.86%")

    st.markdown("---")
    st.warning("⚠️ **Automated System Rule:** If the Technical Agent detects a sudden drop in transaction volume concurrent with a negative moving average crossover or RSI overbought crossover, the **Portfolio Manager Agent** will force the automatic closure of the position to mitigate losses in the fund.")

# 3. INTERACTIVE CHAT (LLM)
with tab_chat:
    st.subheader("💬 Chat Console with Multi-Agent Ecosystem & LLM")
    st.markdown("Converse directly with AI Multi-Agents to generate technical analyses or perform quick queries on DuckDB:")

    for msg in st.session_state.chat_history:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

    if user_prompt := st.chat_input("For example: What is the stance of the 3 agents for Apple today?"):
        with st.chat_message("user"):
            st.markdown(user_prompt)
        st.session_state.chat_history.append({"role": "user", "content": user_prompt})

        try:
            res = requests.post(f"{API_URL}/api/chat", json={"message": user_prompt})
            if res.status_code == 200:
                bot_reply = res.json().get("response", "Unanswered.")
            else:
                bot_reply = "Error processing message with multi-agent system."
        except Exception:
            bot_reply = "⚠️ Critical error: The FastAPI backend (port 8000) is not responding."

        with st.chat_message("assistant"):
            st.markdown(bot_reply)
        st.session_state.chat_history.append({"role": "assistant", "content": bot_reply})

Writing app.py


**5.4 Background Launch**

In [ ]:
# 1. Run the Backend (FastAPI) in a secure background
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 > fastapi.log 2>&1 &
# 2. Run the Frontend (Streamlit) in a secure background
!nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &

# 3. Obtain the public IP address of the Colab runtime environment (Password for the tunnel)
print("\n" + "="*60)
print("🔑 IMPORTANT STEP:")
print("Copy the following numeric IP address. When you open the Localtunnel link,")
print("Paste it into the 'Tunnel Password' field to unlock the Streamlit interface.")
print("="*60)
!curl https://ipv4.icanhazip.com
print("="*60 + "\n")

# 4. Create the exposed secure tunnel to open the Dashboard
!lt --port 8501

**create README**

In [36]:
%%writefile README.md

# Trading Multi-Agent (v1)

An end-to-end Multi-Agent AI Trading Platform designed to extract financial market data, build a structured relational data store using DuckDB, and deploy an intelligent, autonomous AI Multi-Agent powered by Gemini 2.5 Flash to perform technical analysis, query data via tool use (function calling), and generate structured quantitative trading recommendations.

## 📌 Project Architecture & Data Flow

The platform follows a modern **Medallion Architecture** coupled with an **Multi-Agent Layer**:

1. **Bronze Layer (Raw Storage):** Connects to the Yahoo Finance API (`yfinance`) to execute hybrid extraction of both local Colombian Stock Exchange (BVC) assets (e.g., `ECOPETROL.CO`, `ISA.CO`) and major international equities (e.g., `AAPL`, `TSLA`, `NVDA`). Data is safely archived in its raw semi-structured CSV form within a persistent Google Drive Data Lake.
2. **Silver Layer (Structured Relational Store):** Utilizes **DuckDB** to build a high-performance analytical database. Raw CSV index structures are parsed, schema types are enforced, and data is normalized into an idempotent relational model (`dim_assets` and `fact_historical_prices`) using robust `INSERT OR REPLACE` transactions.
3. **Multi-Agent Layer (Intelligence Hub):** Deploys a **Gemini 2.5 Flash** Multi-agent using the modern `google-genai` SDK. Equipped with native Tool Calling (Function Calling), the Multi-agent autonomously determines when and how to generate optimized SQL queries against the DuckDB Silver Layer, analyzes short-term trends, and outputs deterministic financial decisions.

---

## 🛠️ Tech Stack

* **Core AI Engine:** Google GenAI SDK (`google-genai`), `gemini-2.5-flash` (with strict JSON response schemas and system instruction constraints).
* **Database & Analytical Engine:** DuckDB (OLAP-optimized, fast vectorized execution engine, embedded persistence).
* **Data Processing & Engineering:** Pandas, NumPy, Python 3.12.
* **Financial Extraction:** Yahoo Finance API (`yfinance`).
* **Storage / Environment:** Google Colab, Google Drive (Persistent Data Lake Storage).
* **Visualization:** Matplotlib, Seaborn.

---

## 📂 Project Structure & Database Schema

### Relational Model Design

#### 1. `dim_assets` (Dimension Table)
Stores descriptive metadata for all monitored financial tokens.
* `ticker` (VARCHAR, Primary Key) - Cleaned ticker handle.
* `asset_name` (VARCHAR) - Full name of the company or equity.
* `market` (VARCHAR) - Market categorization (`BVC` for Colombian market, `GLOBAL` for international equities).
* `currency` (VARCHAR) - Trading currency (`COP`, `USD`).

#### 2. `fact_historical_prices` (Fact Table)
Stores structured granular pricing history.
* `ticker` (VARCHAR, Primary Key / Foreign Key)
* `date` (DATE, Primary Key) - Trading date calendar indicator.
* `open` (DOUBLE) - Opening price.
* `high` (DOUBLE) - Daily session high.
* `low` (DOUBLE) - Daily session low.
* `close` (DOUBLE) - Final session close price.
* `adj_close` (DOUBLE) - Adjusted close price for dividends/splits.
* `volume` (BIGINT) - Volume of shares traded.
* `ingestion_timestamp` (TIMESTAMP) - Audit tracking indicator for incremental loads.

---

## 🚀 Step-by-Step Setup Guide

### 1. Persistent Storage Setup
Ensure your Google Drive is mounted within Google Colab to preserve extracted raw data and the DuckDB analytical binary file:
```python
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_PATH = "/content/drive/MyDrive/Trading_Agent_POCv1"
os.makedirs(os.path.join(BASE_PATH, "data", "raw"), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, "db"), exist_ok=True))

Writing README.md


In [37]:
%%writefile README1.md
# 📈 AlphaAgent — Multi-Agent Trading Web Application

[![FastAPI](https://img.shields.io/badge/Backend-FastAPI-009688?style=flat-square&logo=fastapi&logoColor=white)](https://fastapi.tiangolo.com/)
[![Streamlit](https://img.shields.io/badge/Frontend-Streamlit-FF4B4B?style=flat-square&logo=streamlit&logoColor=white)](https://streamlit.io/)
[![Deployment - Render](https://img.shields.io/badge/Deploy-Render-46E3B7?style=flat-square&logo=render&logoColor=white)](https://render.com/)
[![Python 3.11](https://img.shields.io/badge/Python-3.11-3776AB?style=flat-square&logo=python&logoColor=white)](https://www.python.org/)

**AlphaAgent** es una plataforma analítica SaaS orientada al mercado financiero que implementa una arquitectura jerárquica multi-agente de Inteligencia Artificial para el análisis técnico y la toma de decisiones de inversión automatizadas. El sistema simula un Comité Corporativo de Inversión dividiendo la carga cognitiva en capas especializadas (Análisis Básico, Análisis Avanzado y Dirección de Portafolio).

## 🔗 Enlaces del Proyecto

* **Repositorio GitHub:** [multi-agent-trading-web-application-WebApp-](https://github.com/jwanxanqak/multi-agent-trading-web-application-WebApp-/tree/main)
* **Aplicación en Vivo (Frontend):** [AlphaAgent Web Dashboard](https://multi-agent-trading-web-application-c44n.onrender.com/)
* **Servicio de Producción (Backend API):** [AlphaAgent API Core](https://multi-agent-trading-web-application.onrender.com/)

---

## 🏗️ Arquitectura del Sistema y Capas Agénticas

La plataforma descentraliza el flujo analítico mediante tres entidades lógicas pre-procesadas en un almacén de caché optimizado (`agent_cache.json`):

1.  **Basic Analysis Agent:** Evalúa tendencias de corto plazo (ventanas de 15 días), calcula rangos promedio de volatilidad diaria y analiza la correlación entre la acción del precio y las divergencias de volumen transaccional básico.
2.  **Advanced Analysis Agent:** Procesa ventanas temporales extendidas e infiere dinámicamente el comportamiento de indicadores técnicos estandarizados como medias móviles simples (SMA_10, SMA_20) y el Índice de Fuerza Relativa (RSI_14) para identificar zonas de sobrecompra o sobreventa.
3.  **Portfolio Manager Agent (Consolidador):** Actúa como el filtro supervisor. Modera las discrepancias analíticas entre los agentes de soporte e integra políticas corporativas para emitir una recomendación final unificada (`BUY`, `HOLD`, `SELL`) con su debida justificación ejecutiva.

---

## ⚡ Características Clave

* **Arquitectura Desacoplada (Decoupled SaaS):** Backend construido sobre una API REST robusta y asíncrona con FastAPI, y un Frontend interactivo de alta disponibilidad implementado con Streamlit.
* **Análisis Técnico Estructurado:** Extracción automatizada de métricas clave, sugerencias detalladas para la generación de gráficos (series temporales, histogramas de volumen) y reportes automatizados en formato JSON.
* **Control de Riesgo y Simulación de Umbrales:** Módulo frontend dedicado al monitoreo visual de métricas de mitigación de riesgo financiero (ej. umbrales de exposición máxima).
* **Consola de Comité Agéntico (Interactive Chat):** Chat interactivo simulado que permite interrogar dinámicamente el criterio técnico del comité de agentes de IA para activos seleccionados.

---

## 🛠️ Stack Tecnológico

* **Backend:** Python 3.11, FastAPI, Pydantic v2 (Validación de tipos de datos de entrada).
* **Frontend:** Streamlit, Requests (Consumo asíncrono de microservicios).
* **Persistencia Analítica:** Caché estructurado JSON optimizado para la eliminación de redundancias de tokens y latencia de inferencia.
* **DevOps / Infraestructura:** Render Blueprint Spec (`render.yaml`), Uvicorn (ASGI Web Server).

---
## 📁 Estructura del Repositorio

```text
├── agent_cache.json   # Base de conocimiento estructurada con análisis técnicos pre-calculados.
├── app.py             # Frontend interactivo construido en Streamlit (Monitor, Riesgo y Chat).
├── main.py            # Orquestador del Backend (FastAPI Core) con enrutamiento de señales y chat.
├── render.yaml        # Infraestructura como Código (IaC) para el despliegue orquestado en la nube.
└── requirements.txt   # Manifiesto de dependencias con versiones fijadas para producción.

Writing README1.md


In [38]:
%%writefile README1.md

# ☁️ WebApp Architecture & Prototypical Deployment

This section covers the implementation of a decoupled microservices architecture designed to serve the Multi-Agent Trading System. The system features a **FastAPI Asynchronous Backend Engine** and a **Streamlit Interactive Dashboard Frontend**, integrated dynamically within a Google Colab ecosystem and securely exposed via a reverse-proxy tunnel (`localtunnel`).

```                 ┌──────────────────────────────────────────────┐
                    │               Google Drive                   │
                    │         (Persistent Data Lake)               │
                    │           └─ agent_cache.json                │
                    └──────────────────────┬───────────────────────┘
                                           │ Read-Only (Poll)
                                           ▼
┌──────────────────────┐        ┌──────────────────────┐        ┌──────────────────────┐
│  Localtunnel Proxy   │        │  FastAPI Core API    │        │ Streamlit UI Engine  │
│  (Public Gateway)    │◄──────►│     (Port 8000)      │◄──────►│     (Port 8501)      │
│                   )  │  HTTP  │  - Signal Endpoints  │  HTTP  │  - Asset Monitor     │
│  [Secured via IP]    │        │  - Conversational LLM│        │  - Risk Management   │
└──────────────────────┘        └──────────────────────┘        └──────────────────────┘

---

## 5.1 Environment Setup & Core Dependencies

To construct the decoupled application layer and bridge local network stacks out of the virtualized environment, we provision Python framework constraints alongside Node.js-based routing networks.

## 5.2 Backend Service Infrastructure (`main.py`)

The backend layer is architected via **FastAPI** to expose high-performance RESTful APIs. It implements strict data validation schemas using `pydantic` and acts as the execution layer for parsing, combining, and orchestrating downstream agent consensus policies.

### Architectural Logic Highlights:
* **Payload Sanitation & Parsing Engine:** Automatically extracts, formats, and sanitizes raw JSON block text (stripping markdown syntax wraps like ` ```json `) cached by previous execution cycles in the Google Drive Data Lake.
* **Agent Consensus & Conflict Resolution Algorithm:** Integrates deterministic logic evaluating the outputs of the **Basic AI Agent** and **Advanced AI Agent**. If an analytical divergence is flagged, the **Portfolio Manager Agent** triggers an override pipeline, favoring the deeper mathematical metrics processed via the DuckDB indexing system.
* **Resilience & Fallback Matrix:** Implements a structured graceful degradation fallback mechanism (`has_data: False`) to catch cache misses or unindexed tickers without throwing breaking HTTP 500 runtime faults.

## 5.3 Interactive Frontend User Interface (`app.py`)

The presentation layer is developed in **Streamlit**, offering an analytical cockpit for algorithmic execution tracking. The view layout uses modern UI modules organized across distinct, componentized tabs.

### Module Configuration:
1. **Dynamic Control Sidebar:** Includes a batch asset processing array containing 30 global and local stock market symbols (`AVAILABLE_ASSETS`). It features a global state macro checkbox (`Select all 30 assets`) that dynamically handles massive lookups without raising race conditions or execution bugs.
2. **Multi-Agent Asset Monitor Tab:** Uses explicit state containers (`st.expander`) and clean async HTTP queries to visually contrast agent reasoning side-by-side using unified `st.metric` indicators and conditional rendering markers (`st.success`, `st.error`, `st.warning`).
3. **Portfolio Monitoring & Risk Rules Tab:** Maps real-time data checkpoints, maximum drawdowns, and structural protection alerts managed down-stream via DuckDB.
4. **Conversational Interactive Console Tab:** Simulates asynchronous agent-to-user chat interaction with stateful history maintenance preserved inside `st.session_state`.

## 5.4 Asynchronous Infrastructure Execution & Network Tunneling

Because Google Colab operates within ephemeral, isolated virtual machines, running network daemons directly requires decoupling standard interactive shell blockades. We handle this via `nohup` (No Hang Up) sub-processes and expose internal ports via a reverse-proxy topology.



Overwriting README1.md


In [39]:
%%writefile README1.md
## 🛠️ Guía de Configuración e Instalación Local

Sigue estos pasos detallados para clonar, instalar y ejecutar el ecosistema completo de **AlphaAgent** en tu entorno local de desarrollo utilizando dos terminales independientes.

### 📋 Requisitos Previos
Asegúrate de tener instalado globalmente en tu sistema:
* **Python 3.11** (o versión superior estable).
* **Git** para el control de versiones.

---

### Paso 1: Clonar el Repositorio
Abre tu terminal y descarga el código fuente del proyecto:
```bash
git clone [https://github.com/jwanxanqak/multi-agent-trading-web-application-WebApp-.git](https://github.com/jwanxanqak/multi-agent-trading-web-application-WebApp-.git)
cd multi-agent-trading-web-application-WebApp-

Overwriting README1.md
